## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

## Overview

This notebook runs the improved cascade path and prepares the final cascade candidate for comparison. It is part of the Gold portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook runs the improved cascade path and prepares the final cascade candidate for comparison.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Connect the modeling or validation outputs back to the final anomaly-detection evidence used in the capstone write-up.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/EDA_Notebook_Pump_Gold_03c_Cascade_Modeling_code_reference.md`


## Stage 3 Improved Cascade Setup and Imports

### Ask

Why is this notebook separate from the tuned cascade notebook?

### Answer

This notebook builds on the tuned cascade but focuses on improving the Stage 3 confirmation layer.

The setup imports the same modeling, artifact, lineage, and tracking utilities used by the other Gold modeling notebooks, but the interpretation of this notebook should be specific to the Stage 3 improved cascade variant.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Mapping, cast
from itertools import product

from pathlib import Path
import yaml

import os
import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import train_test_split, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor


# Custom Utilities Module
from utils.core.paths import get_paths
from utils.core.file_io import load_data, save_data, save_json, load_json
from utils.core.logging_profiler import profile_dataframe
from utils.core.logging_setup import configure_logging, log_layer_paths
from utils.core.wandb_utils import finalize_wandb_stage

from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.core.config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.medallion.gold.cascade_row_tracking import (
    ensure_stable_row_id,
    build_stage_scoring_frame,
    score_isolation_forest_stage,
    merge_stage_results_back,
    finalize_stage_flag_columns,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.medallion.gold.gold_cascade_validation_contracts import (
    build_cascade_variant_contract,
    build_stage3_rule_payload_from_globals,
    write_json_contract,
)

from utils.core.artifacts import gold_model_validation_contract_path
from utils.medallion.gold.gold_cascade_validation_contracts import (
    build_gold_model_output_validation_contract,
    write_gold_model_output_validation_contract,
)


from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)

from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)

# Ledger 
from utils.core.ledger import Ledger

# 
from utils.core.artifacts import (
    build_artifact_dirs_from_config,
    artifact_file_path,
)

from utils.core.notebook_context import load_notebook_context

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

In [2]:
def choose_threshold_by_percentile(
    scores: Sequence[float],
    percentile: float = 95.0,
    *,
    return_info: bool = False,
) -> float | Tuple[float, Dict[str, Any]]:
    """
    Choose anomaly threshold using a score percentile.

    Compatibility behavior
    ----------------------
    - Notebook-style usage:
        threshold = choose_threshold_by_percentile(scores, 95.0)
      returns a float threshold.
    - Pipeline-style usage:
        threshold, info = choose_threshold_by_percentile(scores, 95.0, return_info=True)
      returns both threshold and metadata.
    """
    
    scores_array = np.asarray(scores, dtype=float)

    if scores_array.size == 0:
        raise ValueError("Cannot choose threshold from empty score array.")

    threshold = float(np.percentile(scores_array, float(percentile)))

    info = {
        "threshold_method": "percentile",
        "percentile": float(percentile),
        "threshold": threshold,
        "score_count": int(scores_array.size),
        "score_min": float(np.min(scores_array)),
        "score_max": float(np.max(scores_array)),
        "score_mean": float(np.mean(scores_array)),
    }

    if return_info:
        return threshold, info
    return threshold

#### Define configuration mapping guards

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [3]:
# ----
# 
# ----

def cfg_require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{name} must be a mapping, got {type(value).__name__}: {value!r}"
        )

    return cast(Mapping[str, Any], value)

# ----
# 
# ----

def cfg_optional_mapping(value: object | None, name: str) -> Mapping[str, Any]:
    if value is None:
        return {}

    return cfg_require_mapping(value, name)

# ----
# 
# ----

def require_mapping(value: Any, name: str) -> dict[str, Any]:
    """
    Validate that a loaded JSON/config object is a dictionary.
    """
    if not isinstance(value, dict):
        raise TypeError(
            f"{name} must be a dictionary. "
            f"Got {type(value).__name__}: {value!r}"
        )

    return value

# ----
# 
# ----

def require_str_list(value: Any, name: str) -> list[str]:
    """
    Validate that a loaded JSON/config object is a list of strings.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    if not isinstance(value, (list, tuple)):
        raise TypeError(
            f"{name} must be a list/tuple of column names. "
            f"Got {type(value).__name__}: {value!r}"
        )

    cleaned_values = [
        str(item).strip()
        for item in value
        if str(item).strip()
    ]

    if not cleaned_values:
        raise ValueError(f"{name} resolved to an empty list.")

    return cleaned_values


def require_float(value: Any, name: str) -> float:
    """
    Convert a scalar or threshold-return tuple into a float.

    Some project helpers may return either:
        threshold
    or:
        (threshold, metadata)
    """
    if isinstance(value, tuple):
        if len(value) == 0:
            raise ValueError(f"{name} tuple is empty.")
        value = value[0]

    return float(value)


def as_bool_array(value: Any, name: str) -> np.ndarray:
    """
    Convert a Pandas/NumPy boolean mask into a NumPy bool array.
    """
    if isinstance(value, pd.Series):
        return value.to_numpy(dtype=bool)

    return np.asarray(value, dtype=bool)


def as_int_array(value: Any, name: str) -> np.ndarray:
    """
    Convert labels/flags into a NumPy int array.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    if isinstance(value, pd.Series):
        return value.fillna(0).astype(int).to_numpy(dtype=int)

    return np.asarray(value, dtype=int)


def as_float_array(value: Any, name: str) -> np.ndarray:
    """
    Convert scores into a flat NumPy float array.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    return np.asarray(value, dtype=float).reshape(-1)


def choose_threshold_value(scores: Any, percentile: float) -> float:
    """
    Normalize score input and threshold helper output for Pylance.
    """
    score_values = as_float_array(scores, "scores")
    threshold_result = choose_threshold_by_percentile(
        score_values.tolist(),
        percentile,
    )
    return require_float(threshold_result, "threshold")

---

## Stage 3 Improved Cascade Setup and Imports

### Ask

Why is this notebook separate from the tuned cascade notebook?

### Answer

This notebook builds on the tuned cascade but focuses on improving the Stage 3 confirmation layer.

The setup imports the same modeling, artifact, lineage, and tracking utilities used by the other Gold modeling notebooks, but the interpretation of this notebook should be specific to the Stage 3 improved cascade variant.

In [4]:
# Shared notebook context
CONTEXT_STAGE = "gold_cascade"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "gold"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "gold_modeling_cascade_stage3_improved.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.gold.cascade.stage3_improved",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
GOLD_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-16 20:43:38,737 | INFO | capstone.gold.cascade.stage3_improved | gold_cascade stage starting
2026-06-16 20:43:38,741 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:43:38.741165+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/gold_modeling_cascade_stage3_improved.log'}}
2026-06-16 20:43:38,744 | INFO | capstone.gold.cascade.stage3_improved | Notebook context loaded
2026-06-16 20:43:38,747 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:43:38.747778+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'context_loaded', 'message': 'Loaded shared notebo

{'ts_utc': '2026-06-16T20:43:38.747778+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded shared notebook context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_cascade',
  'recipe_id': 'gold_modeling__v001_cascade',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/gold_modeling_cascade_stage3_improved.log'}}

In [5]:

# Improved Stage 3 cascade notebook selector.
CASCADE_VARIANT = "stage3_improved"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

TRUTH_CONFIG = cast(Dict[str, Any], build_truth_config_block(CONFIG))
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Stage details ----

STAGE = "gold"

LAYER_NAME = str(GOLD_CFG["layer_name"])

RECIPE_ID = str(GOLD_CFG["recipe_id"])

GOLD_VERSION = str(VERSIONS_CFG["gold"])
TRUTH_VERSION = str(VERSIONS_CFG["truth"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

PIPELINE_MODE = str(PIPELINE["execution_mode"])
RUN_MODE = str(RUNTIME_CFG.get("mode", CONFIG_RUN_MODE))
CONFIG_PROFILE = str(RUNTIME_CFG.get("profile", CONFIG_PROFILE))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

DATASET_NAME_CONFIG = str(DATASET_CFG.get("name", "pump"))
DATASET_NAME = DATASET_NAME_CONFIG.strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

GOLD_PROCESS_RUN_ID = make_process_run_id(
    str(GOLD_CFG["process_run_id_prefix"])
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Cascade config blocks ----

STAGE1_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage1", {}), "CONFIG['gold_cascade']['stage1']"),
)

STAGE2_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage2", {}), "CONFIG['gold_cascade']['stage2']"),
)

STAGE3_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage3", {}), "CONFIG['gold_cascade']['stage3']"),
)

STAGE2_FIXED_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_CFG.get("fixed", {}),
        "CONFIG['gold_cascade']['stage2']['fixed']",
    ),
)

STAGE2_SEARCH_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_CFG.get("search", {}),
        "CONFIG['gold_cascade']['stage2']['search']",
    ),
)

STAGE2_FIXED_PARAMS = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_FIXED_CFG.get("params", {}),
        "CONFIG['gold_cascade']['stage2']['fixed']['params']",
    ),
)

STAGE2_SEARCH_PARAM_GRID_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_SEARCH_CFG.get("param_grid", {}),
        "CONFIG['gold_cascade']['stage2']['search']['param_grid']",
    ),
)

STAGE3_TUNING_GRID = cast(
    Dict[str, Any],
    cfg_optional_mapping(
        STAGE3_CFG.get("stage3_tuning_grid"),
        "CONFIG['gold_cascade']['stage3']['stage3_tuning_grid']",
    ),
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Runtime knobs ----

RANDOM_SEED = int(GOLD_CFG["random_seed"])

STAGE1_ESTIMATOR_COUNT = int(STAGE1_CFG["estimator_count"])
STAGE1_THRESHOLD_PERCENTILE = float(STAGE1_CFG["threshold_percentile"])

# Stage 2.
STAGE2_SELECTION_SOURCE = str(
    STAGE2_CFG.get("selection_source", "auto")
).strip().lower()

STAGE2_SELECTION_MODE = str(
    STAGE2_CFG.get("selection_mode", "parameter_search")
).strip().lower()

STAGE2_RANDOM_STATE = int(STAGE2_CFG.get("random_state", RANDOM_SEED))
STAGE2_MIN_RECALL = float(STAGE2_CFG.get("min_recall", 0.50))

STAGE2_FIXED_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG["threshold_percentile"]
)

STAGE2_WARNING_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG.get(
        "warning_threshold_percentile",
        STAGE2_FIXED_THRESHOLD_PERCENTILE,
    )
)

STAGE2_CONFIRMED_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG.get(
        "confirmed_threshold_percentile",
        STAGE2_FIXED_THRESHOLD_PERCENTILE,
    )
)

STAGE2_THRESHOLD_GRID = [
    float(value)
    for value in STAGE2_SEARCH_CFG["threshold_grid"]
]

STAGE2_PARAM_GRID = {
    str(key): list(value)
    for key, value in STAGE2_SEARCH_PARAM_GRID_CFG.items()
}

# Improved Stage 3 settings.
STAGE3_MIN_PRIMARY_SENSOR_HITS = int(
    STAGE3_CFG.get("min_primary_sensor_hits", 2)
)

STAGE3_MIN_SECONDARY_SENSOR_HITS = int(
    STAGE3_CFG.get("min_secondary_sensor_hits", 1)
)

STAGE3_ROLLING_WINDOW_SIZE = int(
    STAGE3_CFG.get("rolling_window_size", 3)
)

STAGE3_MINIMUM_FLAGS_IN_WINDOW = int(
    STAGE3_CFG.get("minimum_flags_in_window", 2)
)

STAGE3_MIN_WEIGHTED_SCORE = float(
    STAGE3_TUNING_GRID.get("min_weighted_score", [3.25])[0]
)

STAGE3_STRONG_PRIMARY_HITS = int(
    STAGE3_TUNING_GRID.get("strong_primary_hits", [3])[0]
)

STAGE3_DRIFT_THRESHOLD_MULTIPLIER = float(
    STAGE3_TUNING_GRID.get("drift_threshold_multiplier", [1.0])[0]
)

STAGE3_DRIFT_ROLLING_WINDOW_SIZE = int(
    STAGE3_CFG.get("drift_rolling_window_size", 5)
)

STAGE3_MIN_SELECTION_RECALL = float(
    STAGE3_CFG.get("min_selection_recall", 0.50)
)

STAGE3_PROFILE_BREACH_WEIGHT = float(
    STAGE3_CFG.get("profile_breach_weight", 2.0)
)

STAGE3_CORROBORATION_WEIGHT = float(
    STAGE3_CFG.get("corroboration_weight", 1.0)
)

STAGE3_PERSISTENCE_WEIGHT = float(
    STAGE3_CFG.get("persistence_weight", 1.0)
)

STAGE3_DRIFT_WEIGHT = float(
    STAGE3_CFG.get("drift_weight", 1.0)
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- W&B ----

WANDB_PROJECT = str(WANDB_CFG.get("project", "capstone"))
WANDB_ENTITY = str(WANDB_CFG.get("entity", ""))
WANDB_RUN_NAME = f"{GOLD_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- File names ----

GOLD_PREPROCESSED_FILE_NAME = str(FILENAMES["gold_preprocessed_file_name"])
GOLD_PREPROCESSED_SCALED_FILE_NAME = str(
    FILENAMES["gold_preprocessed_scaled_file_name"]
)

GOLD_FIT_FILE_NAME = str(FILENAMES["gold_fit_file_name"])
GOLD_TRAIN_FILE_NAME = str(FILENAMES["gold_train_file_name"])
GOLD_TEST_FILE_NAME = str(FILENAMES["gold_test_file_name"])

STAGE1_FEATURES_FILE_NAME = str(FILENAMES["stage1_features_file_name"])
STAGE2_FEATURES_FILE_NAME = str(FILENAMES["stage2_features_file_name"])
STAGE3_PRIMARY_FILE_NAME = str(FILENAMES["stage3_primary_file_name"])
STAGE3_SECONDARY_FILE_NAME = str(FILENAMES["stage3_secondary_file_name"])

STAGE1_MODEL_FILE_NAME = str(
    FILENAMES["cascade_stage3_improved_stage1_model_file_name"]
)
STAGE2_MODEL_FILE_NAME = str(
    FILENAMES["cascade_stage3_improved_stage2_model_file_name"]
)

CASCADE_RESULTS_FILE_NAME_CSV = str(
    FILENAMES["cascade_stage3_improved_results_file_name_csv"]
)
CASCADE_RESULTS_FILE_NAME_PICKLE = str(
    FILENAMES["cascade_stage3_improved_results_file_name_pickle"]
)

CASCADE_THRESHOLDS_FILE_NAME = str(
    FILENAMES["cascade_stage3_improved_thresholds_file_name"]
)
CASCADE_SUMMARY_FILE_NAME = str(
    FILENAMES["cascade_stage3_improved_summary_file_name"]
)
CASCADE_METADATA_FILE_NAME = str(
    FILENAMES["cascade_stage3_improved_metadata_file_name"]
)

CASCADE_REFERENCE_PROFILE_FILE_NAME = str(
    FILENAMES.get(
        "cascade_stage3_improved_reference_profile_file_name",
        f"{DATASET_NAME}__gold__cascade_stage3_improved_reference_profile.csv",
    )
)

CASCADE_TUNED_THRESHOLDS_FILE_NAME = str(
    FILENAMES["cascade_tuned_thresholds_file_name"]
)
CASCADE_TUNED_SUMMARY_FILE_NAME = str(
    FILENAMES["cascade_tuned_summary_file_name"]
)

GOLD_CASCADE_STAGE3_IMPROVED_LEDGER_FILE_NAME = str(
    FILENAMES.get(
        "gold_cascade_stage3_improved_ledger_file_name",
        "gold_cascade_stage3_improved_ledger.jsonl",
    )
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Base paths only ----

ARTIFACTS_ROOT = Path(str(RESOLVED_PATHS["artifacts_root"]))

GOLD_ARTIFACTS_PATH = Path(str(RESOLVED_PATHS["gold_artifacts_dir"]))

GOLD_PREPROCESSED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_data_path"])
)
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_scaled_data_path"])
)

GOLD_FIT_DATA_PATH = Path(str(RESOLVED_PATHS["gold_fit_data_path"]))
GOLD_TRAIN_DATA_PATH = Path(str(RESOLVED_PATHS["gold_train_data_path"]))
GOLD_TEST_DATA_PATH = Path(str(RESOLVED_PATHS["gold_test_data_path"]))

STAGE1_FEATURES_PATH = Path(str(RESOLVED_PATHS["stage1_features_path"]))
STAGE2_FEATURES_PATH = Path(str(RESOLVED_PATHS["stage2_features_path"]))
STAGE3_PRIMARY_PATH = Path(str(RESOLVED_PATHS["stage3_primary_path"]))
STAGE3_SECONDARY_PATH = Path(str(RESOLVED_PATHS["stage3_secondary_path"]))

MODELS_PATH = Path(str(RESOLVED_PATHS["models_root"]))

STAGE1_MODELS_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_stage1_models_path"])
)
STAGE1_MODEL_ARTIFACT_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_stage1_model_artifact_path"])
)

STAGE2_MODELS_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_stage2_models_path"])
)
STAGE2_MODEL_ARTIFACT_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_stage2_model_artifact_path"])
)

CASCADE_RESULTS_PATH_CSV = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_results_path_csv"])
)
CASCADE_RESULTS_PATH_PICKLE = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_results_path_pickle"])
)

CASCADE_THRESHOLDS_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_thresholds_path"])
)
CASCADE_SUMMARY_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_summary_path"])
)
CASCADE_METADATA_PATH = Path(
    str(RESOLVED_PATHS["cascade_stage3_improved_metadata_path"])
)

CASCADE_REFERENCE_PROFILE_PATH = Path(
    str(
        RESOLVED_PATHS.get(
            "cascade_stage3_improved_reference_profile_path",
            GOLD_ARTIFACTS_PATH / CASCADE_REFERENCE_PROFILE_FILE_NAME,
        )
    )
)

CASCADE_TUNED_THRESHOLDS_PATH = Path(
    str(RESOLVED_PATHS["cascade_tuned_thresholds_path"])
)
CASCADE_TUNED_SUMMARY_PATH = Path(
    str(RESOLVED_PATHS["cascade_tuned_summary_path"])
)

TRUTHS_PATH = Path(str(RESOLVED_PATHS["truths_dir"]))
TRUTH_INDEX_PATH = Path(str(RESOLVED_PATHS["truth_index_path"]))

LOGS_PATH = Path(str(RESOLVED_PATHS["logs_root"]))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# W&B

set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# Path failsafes

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

STAGE1_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE2_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE3_PRIMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE3_SECONDARY_PATH.parent.mkdir(parents=True, exist_ok=True)

MODELS_PATH.mkdir(parents=True, exist_ok=True)

STAGE1_MODELS_PATH.parent.mkdir(parents=True, exist_ok=True)
#STAGE1_MODEL_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

STAGE2_MODELS_PATH.parent.mkdir(parents=True, exist_ok=True)
#STAGE2_MODEL_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

CASCADE_RESULTS_PATH_CSV.parent.mkdir(parents=True, exist_ok=True)
CASCADE_RESULTS_PATH_PICKLE.parent.mkdir(parents=True, exist_ok=True)

CASCADE_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)
CASCADE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
CASCADE_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

CASCADE_REFERENCE_PROFILE_PATH.parent.mkdir(parents=True, exist_ok=True)

CASCADE_TUNED_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)
CASCADE_TUNED_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

print("Project root:", paths.root)
print("Artifacts root:", ARTIFACTS_ROOT)
print("CONFIG dataset:", DATASET_NAME_CONFIG)
print("Cascade variant:", CASCADE_VARIANT)
print("Stage 2 selection source:", STAGE2_SELECTION_SOURCE)
print("Stage 2 selection mode:", STAGE2_SELECTION_MODE)
print("Gold fit path:", GOLD_FIT_DATA_PATH)
print("Stage 1 model path:", STAGE1_MODEL_ARTIFACT_PATH)
print("Stage 2 model path:", STAGE2_MODEL_ARTIFACT_PATH)
print("Cascade results path:", CASCADE_RESULTS_PATH_CSV)
print("Cascade summary path:", CASCADE_SUMMARY_PATH)

Project root: /workspace
Artifacts root: /workspace/artifacts
CONFIG dataset: pump
Cascade variant: stage3_improved
Stage 2 selection source: auto
Stage 2 selection mode: parameter_search
Gold fit path: /workspace/data/gold/pump__gold__fit_normal_only.parquet
Stage 1 model path: /workspace/artifacts/gold/pump/cascade_stage3_improved/models/pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib
Stage 2 model path: /workspace/artifacts/gold/pump/cascade_stage3_improved/models/pump__gold__cascade_stage3_improved_stage2_isolation_forest.joblib
Cascade results path: /workspace/artifacts/gold/pump/cascade_stage3_improved/scores/pump__gold__cascade_stage3_improved_results.csv
Cascade summary path: /workspace/artifacts/gold/pump/cascade_stage3_improved/summaries/pump__gold__cascade_stage3_improved_summary.json


----

In [6]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required shared context variables: {missing_context_vars}")

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame(
    [
        {"name": name, "status": "present"}
        for name in required_context_vars
    ]
)

2026-06-16 20:43:39,131 | INFO | capstone.gold.cascade.stage3_improved | Context sanity check passed
2026-06-16 20:43:39,133 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:43:39.133390+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'check', 'step': 'context_sanity_check', 'message': 'Verified shared notebook context variables are available.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/gold_modeling_cascade_stage3_improved.log'}}


,name,status
0,CTX,present
1,paths,present
2,CONFIG,present
3,CONFIG_MAP,present
4,STAGE_CFG,present
5,RESOLVED_PATHS,present
6,FILENAMES,present
7,VERSIONS_CFG,present
8,RUNTIME_CFG,present
9,DATASET_CFG,present


In [7]:
gold_required_context_vars = [
    "GOLD_CFG",
]

missing_gold_context_vars = [
    name for name in gold_required_context_vars
    if name not in globals()
]

if missing_gold_context_vars:
    raise NameError(f"Missing Gold context variables: {missing_gold_context_vars}")

logger.info("Gold context sanity check passed")

2026-06-16 20:43:39,173 | INFO | capstone.gold.cascade.stage3_improved | Gold context sanity check passed


---

In [8]:
CASCADE_VARIANT = "stage3_improved"

GOLD_CASCADE_ARTIFACT_DIRS = build_artifact_dirs_from_config(
    config=CONFIG,
    stage_key="gold_cascade",
    variant=CASCADE_VARIANT,
)

GOLD_ARTIFACTS_PATH = GOLD_CASCADE_ARTIFACT_DIRS["stage_dataset_root"]
GOLD_CASCADE_ROOT = GOLD_CASCADE_ARTIFACT_DIRS["root"]

CASCADE_RESULTS_PATH_CSV = (
    GOLD_CASCADE_ARTIFACT_DIRS["scores"]
    / FILENAMES["cascade_stage3_improved_results_file_name_csv"]
)

CASCADE_RESULTS_PATH_PICKLE = (
    GOLD_CASCADE_ARTIFACT_DIRS["scores"]
    / FILENAMES["cascade_stage3_improved_results_file_name_pickle"]
)

CASCADE_THRESHOLDS_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["thresholds"]
    / FILENAMES["cascade_stage3_improved_thresholds_file_name"]
)

CASCADE_SUMMARY_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["summaries"]
    / FILENAMES["cascade_stage3_improved_summary_file_name"]
)

CASCADE_METADATA_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["metadata"]
    / FILENAMES["cascade_stage3_improved_metadata_file_name"]
)

CASCADE_REFERENCE_PROFILE_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["profiles"]
    / FILENAMES["cascade_stage3_improved_reference_profile_file_name"]
)

STAGE1_MODEL_ARTIFACT_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["models"]
    / FILENAMES["cascade_stage3_improved_stage1_model_file_name"]
)

STAGE2_MODEL_ARTIFACT_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["models"]
    / FILENAMES["cascade_stage3_improved_stage2_model_file_name"]
)

#STAGE1_MODELS_PATH = STAGE1_MODEL_ARTIFACT_PATH
#STAGE2_MODELS_PATH = STAGE2_MODEL_ARTIFACT_PATH

cascade_ledger_path = (
    GOLD_CASCADE_ARTIFACT_DIRS["lineage"]
    / FILENAMES["gold_cascade_stage3_improved_ledger_file_name"]
)

GOLD_CASCADE_CONFIG_DIR = GOLD_CASCADE_ARTIFACT_DIRS["config"]

CONFIG_SNAPSHOT_PATH = (
    GOLD_CASCADE_CONFIG_DIR
    / f"{DATASET_NAME}__gold_cascade_stage3_improved__resolved_config.yaml"
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

---

In [9]:
# =============================================================================
# SQL Runtime Context
# Purpose:
#   Establish the Postgres connection and resolve the dataset/run identifiers
#   used by SQL logging and medallion table writes.
# =============================================================================

engine = get_engine_from_env()

CAPSTONE_SCHEMA: str = str(os.getenv("CAPSTONE_SCHEMA", "capstone"))


def first_non_empty_string(*values: object) -> str | None:
    """
    Return the first non-empty string-like value from a list of candidates.

    This helper skips None, empty strings, whitespace-only strings, and
    dictionaries. It is used to resolve dataset/run identifiers without
    accidentally accepting an invalid config object or a placeholder None value.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, dict):
            continue

        text_value = str(value).strip()

        if text_value:
            return text_value

    return None


dataset_config = (
    cast(Dict[str, Any], CONFIG.get("dataset", {}))
    if isinstance(CONFIG.get("dataset", {}), dict)
    else {}
)

dataset_config_name = dataset_config.get("name")
dataset_config_id = dataset_config.get("dataset_id", dataset_config.get("id"))
dataset_config_run_id = dataset_config.get("run_id")
dataset_config_asset_id = dataset_config.get("asset_id")

is_synthetic_run = str(RUN_MODE).lower() in {
    "synthetic",
    "synthetic_train",
    "synthetic_run",
    "simulation",
}

if is_synthetic_run:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        os.getenv("SYNTHETIC_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump_synthetic_v1",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        os.getenv("SYNTHETIC_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "synthetic_run_001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        os.getenv("SYNTHETIC_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "pump_asset_001",
    )

else:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "run__001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "asset__001",
    )


if raw_dataset_id is None:
    raise ValueError(
        "DATASET_ID could not be resolved. "
        "Set CAPSTONE_DATASET_ID or configure CONFIG['dataset']['name'] / "
        "CONFIG['dataset']['dataset_id']."
    )

if raw_run_id is None:
    raise ValueError(
        "RUN_ID could not be resolved. "
        "Set CAPSTONE_RUN_ID, CONFIG['dataset']['run_id'], or default_fallbacks.run_id."
    )

if raw_asset_id is None:
    raise ValueError(
        "ASSET_ID could not be resolved. "
        "Set CAPSTONE_ASSET_ID, CONFIG['dataset']['asset_id'], or default_fallbacks.asset_id."
    )


DATASET_ID: str = raw_dataset_id
RUN_ID: str = raw_run_id
ASSET_ID: str = raw_asset_id


print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")
print(f"Asset ID: {ASSET_ID}")
print(f"Synthetic run: {is_synthetic_run}")

SQL schema: capstone
Dataset ID: pump
Run ID: run__001
Asset ID: asset__001
Synthetic run: False


#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [10]:

# =============================================================================
# SQL Smoke Check
# Purpose:
#   Confirm the database connection and expected capstone schemas/tables exist.
# =============================================================================

sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


---

## Start Logging for the Tuned Cascade Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, traceability, and basic pipeline discipline. If I need to check what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [11]:
log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Project paths logged for %s layer",
    CONTEXT_LAYER,
    extra={"stage": CONTEXT_STAGE, "layer": CONTEXT_LAYER},
)

ledger.add(
    kind="step",
    step=f"{CONTEXT_LAYER}_paths_logged",
    message="Logged project layer paths.",
    data={
        "stage": CONTEXT_STAGE,
        "layer": CONTEXT_LAYER,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-16 20:43:39,672 | INFO | capstone.gold.cascade.stage3_improved | Project Root Path Loaded: /workspace
2026-06-16 20:43:39,675 | INFO | capstone.gold.cascade.stage3_improved | Project Logging Path Loaded: /workspace/logs
2026-06-16 20:43:39,677 | INFO | capstone.gold.cascade.stage3_improved | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-16 20:43:39,679 | INFO | capstone.gold.cascade.stage3_improved | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-16 20:43:39,680 | INFO | capstone.gold.cascade.stage3_improved | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-16 20:43:39,682 | INFO | capstone.gold.cascade.stage3_improved | Project Data Path Loaded: /workspace/data
2026-06-16 20:43:39,687 | INFO | capstone.gold.cascade.stage3_improved | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-06-16 20:43:39,690 | INFO | capstone.gold.cascade.stage3_improved | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/tra

{'ts_utc': '2026-06-16T20:43:39.700970+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'gold_paths_logged',
 'message': 'Logged project layer paths.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_cascade',
  'layer': 'gold',
  'log_path': '/workspace/logs/gold_modeling_cascade_stage3_improved.log'}}

In [12]:
""" 
# Original Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade_stage3_improved.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)
 """

' \n# Original Logging Setup\n\n# Create gold log path \ngold_log_path = paths.logs / "gold_modeling_cascade_stage3_improved.log"\n\n# Initial Logger\nconfigure_logging(\n    "capstone",\n    gold_log_path,\n    level=logging.DEBUG,\n    overwrite_handlers=True,\n)\n\n# Initiate Logger and log file\nlogger = logging.getLogger("capstone.gold")\n\n# Log load and initiation\nlogger.info("Gold Modeling stage starting")\n\n# Log paths loads\nlog_layer_paths(paths, current_layer="gold", logger=logger)\n '

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the tuned cascade modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the cascade configuration, inputs, and saved outputs, but it does not change the underlying cascade logic itself.

In [13]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "cascade_variant": CASCADE_VARIANT,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_fixed_params": STAGE2_FIXED_PARAMS,
        "stage2_fixed_threshold_percentile": STAGE2_FIXED_THRESHOLD_PERCENTILE,
        "stage3_strong_primary_hits": STAGE3_STRONG_PRIMARY_HITS,
        "stage3_min_primary_sensor_hits": STAGE3_MIN_PRIMARY_SENSOR_HITS,
        "stage3_min_secondary_sensor_hits": STAGE3_MIN_SECONDARY_SENSOR_HITS,
        "stage3_profile_breach_weight": STAGE3_PROFILE_BREACH_WEIGHT,
        "stage3_corroboration_weight": STAGE3_CORROBORATION_WEIGHT,
        "stage3_persistence_weight": STAGE3_PERSISTENCE_WEIGHT,
        "stage3_drift_weight": STAGE3_DRIFT_WEIGHT,
        "stage3_min_weighted_score": STAGE3_MIN_WEIGHTED_SCORE,
        "stage3_rolling_window_size": STAGE3_ROLLING_WINDOW_SIZE,
        "stage3_minimum_flags_in_window": STAGE3_MINIMUM_FLAGS_IN_WINDOW,
        "stage3_drift_rolling_window_size": STAGE3_DRIFT_ROLLING_WINDOW_SIZE,
        "stage3_drift_threshold_multiplier": STAGE3_DRIFT_THRESHOLD_MULTIPLIER,
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb_run.name)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-06-16 20:43:45,665 | INFO | capstone.gold.cascade.stage3_improved | W&B initialized: gold__001


----

## Initialize the Cascade Ledger

Here I create the ledger that tracks the main steps taken during the tuned cascade notebook.

I treat the ledger as a structured record of the run. It gives me a cleaner summary of the workflow than relying only on printed notebook output, especially when I need to review or compare cascade runs later.

In [14]:
# Ledger was initialized by load_notebook_context().
# Keep this cell as a visible notebook checkpoint instead of re-creating Ledger.

ledger.add(
    kind="step",
    step="ledger_context_ready",
    message="Ledger is available from shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)

logger.info(
    "Ledger ready from notebook context",
    extra={"stage": CONTEXT_STAGE, "recipe_id": CONTEXT_RECIPE_ID},
)


2026-06-16 20:43:46,318 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:43:46.318771+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'ledger_context_ready', 'message': 'Ledger is available from shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'log_path': '/workspace/logs/gold_modeling_cascade_stage3_improved.log'}}
2026-06-16 20:43:46,322 | INFO | capstone.gold.cascade.stage3_improved | Ledger ready from notebook context


In [15]:
""" 
# Original Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)
 """

' \n# Original Ledger Setup\n\nledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)\n\nledger.add(\n    kind="step",\n    step="init",\n    message="Initialized ledger",\n    logger=logger\n)\n '

----

## Load the Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the tuned cascade depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- updating artifact paths from the truth record when available
- loading the Gold fit, train, and test dataframes
- loading the Stage 1, Stage 2, and Stage 3 feature artifacts

This matters because the tuned cascade notebook should inherit the prepared Gold artifacts rather than rebuilding those inputs on its own.

In [16]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold cascade input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

GOLD_PREPROCESSED_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_preprocessed_path", str(GOLD_PREPROCESSED_DATA_PATH)))
GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
GOLD_TEST_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_test_path", str(GOLD_TEST_DATA_PATH)))
GOLD_TRAIN_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_train_path", str(GOLD_TRAIN_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))
STAGE2_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage2_features_path", str(STAGE2_FEATURES_PATH)))
STAGE3_PRIMARY_PATH = Path(gold_truth_artifact_paths.get("stage3_primary_path", str(STAGE3_PRIMARY_PATH)))
STAGE3_SECONDARY_PATH = Path(gold_truth_artifact_paths.get("stage3_secondary_path", str(STAGE3_SECONDARY_PATH)))

logger.info("Resolved Gold cascade dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)

print("Gold cascade dataset name from parent truth:", DATASET_NAME)
print("Gold cascade parent truth hash:", GOLD_PARENT_TRUTH_HASH)

logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_DATA_PATH)
gold_preprocessed_dataframe = load_data(GOLD_PREPROCESSED_DATA_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Gold train parquet: %s", GOLD_TRAIN_DATA_PATH)
gold_train_dataframe = load_data(GOLD_TRAIN_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns: list[str] = require_str_list(
    load_json(STAGE1_FEATURES_PATH),
    "stage1_feature_columns",
)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
stage2_feature_columns: list[str] = require_str_list(
    load_json(STAGE2_FEATURES_PATH),
    "stage2_feature_columns",
)

logger.info("Loading Stage 3 primary rule sensors: %s", STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors: list[str] = require_str_list(
    load_json(STAGE3_PRIMARY_PATH),
    "stage3_primary_rule_sensors",
)

logger.info("Loading Stage 3 secondary rule sensors: %s", STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors: list[str] = require_str_list(
    load_json(STAGE3_SECONDARY_PATH),
    "stage3_secondary_rule_sensors",
)

logger.info("Loading Tuned Stage 2 Parameters: %s", CASCADE_TUNED_THRESHOLDS_PATH)
cascade_tuned_thresholds: dict[str, Any] = require_mapping(
    load_json(CASCADE_TUNED_THRESHOLDS_PATH),
    "cascade_tuned_thresholds",
)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded cascade inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_path": str(GOLD_PREPROCESSED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_train_path": str(GOLD_TRAIN_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_preprocessed_shape": list(gold_preprocessed_dataframe.shape),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "gold_train_shape": list(gold_train_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

display(gold_test_dataframe.head(3))

2026-06-16 20:43:47,070 | INFO | capstone.gold.cascade.stage3_improved | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-06-16 20:43:47,073 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-06-16 20:43:50,610 | INFO | capstone.gold.cascade.stage3_improved | Resolved Gold cascade dataset name from Gold truth: pump
2026-06-16 20:43:50,613 | INFO | capstone.gold.cascade.stage3_improved | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5.json
2026-06-16 20:43:50,620 | INFO | capstone.gold.cascade.stage3_improved | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet
2026-06-16 20:43:50,629 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet


Gold cascade dataset name from parent truth: pump
Gold cascade parent truth hash: 1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5


2026-06-16 20:43:53,022 | INFO | capstone.gold.cascade.stage3_improved | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-06-16 20:43:53,026 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-06-16 20:43:53,930 | INFO | capstone.gold.cascade.stage3_improved | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-06-16 20:43:53,934 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-06-16 20:43:54,768 | INFO | capstone.gold.cascade.stage3_improved | Loading Gold train parquet: /workspace/data/gold/pump__gold__train.parquet
2026-06-16 20:43:54,772 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-06-16 20:43:56,574 | INFO | capstone.gold.cascade.stage3_improved | Loading Stage 1 features: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage1_features.json
2026-06

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_43__value_abnormal_flag,sensor_43__delta_abnormal_flag,sensor_43__any_abnormal_flag,sensor_44__value_deviation,sensor_44__delta_deviation,sensor_44__value_abnormal_flag,sensor_44__delta_abnormal_flag,sensor_44__any_abnormal_flag,sensor_45__value_deviation,sensor_45__delta_deviation,sensor_45__value_abnormal_flag,sensor_45__delta_abnormal_flag,sensor_45__any_abnormal_flag,sensor_46__value_deviation,sensor_46__delta_deviation,sensor_46__value_abnormal_flag,sensor_46__delta_abnormal_flag,sensor_46__any_abnormal_flag,sensor_47__value_deviation,sensor_47__delta_deviation,sensor_47__value_abnormal_flag,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag
0,asset__001,pump,5,pump:asset__001:run__001:136431,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,9701747606095593835,run__001,sensor.csv,136431,test,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-07-04 17:51:00+00:00,136431,136431,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-43.760725,-2.652173,-5.732155,-1.192304,-53.278556,-9.774008,-6.543561,-7.343809,-6.298183,-22.668203,-7.049292,-5.030354,-5.814053,-1.092581,0.009936,-0.247708,0.120209,0.028814,-2.195752,-0.353799,0.251724,0.801197,4.881352,0.050529,0.596592,0.254524,1.055766,0.386601,-2.744446,...,False,False,False,1.25641,NaN,False,False,False,1.175675,NaN,False,False,False,1.176471,NaN,False,False,False,1.391304,NaN,False,False,False,1.760174,NaN,False,False,False,3.170731,NaN,False,False,False,5.428565,NaN,False,False,False,13,0,13,suspect,False,suspect,False,True,True,normal_suspect,136431,False
1,asset__001,pump,5,pump:asset__001:run__001:136432,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,4242984989007806471,run__001,sensor.csv,136432,test,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-07-04 17:52:00+00:00,136432,136432,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-43.760726,-2.456519,-5.732155,-1.230767,-53.085787,-9.836153,-6.652261,-7.453193,-6.298183,-22.668203,-7.076084,-5.030354,-5.814053,-1.092581,0.518691,-1.254584,0.560349,0.546779,-0.798474,-0.303199,0.337026,0.834646,4.842846,0.023142,0.661646,0.255902,0.713836,0.563540,-2.522376,...,True,False,True,1.25641,0.0,False,False,False,1.283782,0.399997,False,False,False,1.176470,0.000002,False,False,False,1.391304,0.0,False,False,False,1.760174,2.111239e-07,False,False,False,3.170731,0.0,False,False,False,5.428565,0.0,False,False,False,14,0,14,suspect,False,suspect,False,True,True,normal_suspect,136432,False
2,asset__001,pump,5,pump:asset__001:run__001:136433,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31da

----

## Verify Stage 3 Improved Input Lineage

### Ask

Why do I check lineage again here?

### Answer

The Stage 3 improved notebook depends on the same Gold preprocessing output used by the baseline, default cascade, and tuned cascade notebooks.

This check confirms that the parent Gold truth hash is available before the Stage 3 improved truth record is created. That helps avoid accidentally saving a Stage 3 improved artifact that is not linked back to the correct Gold preprocessing run.

In [17]:
# =========================================================
# Gold 03c input lineage verification
# =========================================================

print("=== Gold 03c Input Lineage Verification ===")

# In 03c, the upstream Gold truth hash is the parent/input hash.
# CASCADE_TRUTH_HASH is created later after this notebook builds its own output truth record.
gold_parent_truth_hash_value = globals().get("GOLD_PARENT_TRUTH_HASH", "MISSING")
cascade_truth_hash_value = globals().get("CASCADE_TRUTH_HASH", "NOT_CREATED_YET")

print(f"GOLD_PARENT_TRUTH_HASH: {gold_parent_truth_hash_value}")
print(f"CASCADE_TRUTH_HASH: {cascade_truth_hash_value}")


gold_preprocessed_scaled_dataframe_object = globals().get(
    "gold_preprocessed_scaled_dataframe"
)

if isinstance(gold_preprocessed_scaled_dataframe_object, pd.DataFrame):
    if "meta__parent_truth_hash" in gold_preprocessed_scaled_dataframe_object.columns:
        unique_parent_truth_hashes = (
            gold_preprocessed_scaled_dataframe_object["meta__parent_truth_hash"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        print(
            "Unique dataframe meta__parent_truth_hash values: "
            f"{unique_parent_truth_hashes}"
        )
    else:
        print(
            "meta__parent_truth_hash column is missing from "
            "gold_preprocessed_scaled_dataframe"
        )
else:
    print("gold_preprocessed_scaled_dataframe is not loaded")


gold_truth_object = globals().get("gold_truth")

if isinstance(gold_truth_object, dict):
    print("gold_truth runtime/artifact lineage fields:")
    print(
        {
            "truth_hash": gold_truth_object.get("truth_hash"),
            "parent_truth_hash": gold_truth_object.get("parent_truth_hash"),
            "dataset_name": gold_truth_object.get("dataset_name"),
        }
    )
else:
    print("gold_truth object not available")

=== Gold 03c Input Lineage Verification ===
GOLD_PARENT_TRUTH_HASH: 1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5
CASCADE_TRUTH_HASH: NOT_CREATED_YET
Unique dataframe meta__parent_truth_hash values: ['6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3']
gold_truth runtime/artifact lineage fields:
{'truth_hash': '1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5', 'parent_truth_hash': '6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3', 'dataset_name': 'pump'}


#### Resolve the parent truth hash for Gold lineage

This cell defines helper logic used by the surrounding `Verify Stage 3 Improved Input Lineage` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [18]:
def resolve_single_parent_gold_truth_hash(
    dataframe: pd.DataFrame | None = None,
    *,
    dataframe_parent_column: str = "meta__parent_truth_hash",
    truth_record: dict | None = None,
    truth_parent_key: str = "parent_truth_hash",
) -> str:
    """
    Resolve one parent Gold truth hash from the currently loaded Gold inputs.

    Priority
    --------
    1. Use the unique non-null value from the dataframe lineage column.
    2. Fall back to the loaded truth record parent hash.
    """
    dataframe_parent_hashes = []

    if dataframe is not None and dataframe_parent_column in dataframe.columns:
        dataframe_parent_hashes = (
            dataframe[dataframe_parent_column]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

    if len(dataframe_parent_hashes) > 1:
        raise ValueError(
            "Gold 03c input dataframe contains multiple parent Gold truth hashes.\n"
            f"Observed parent hashes: {dataframe_parent_hashes}"
        )

    if len(dataframe_parent_hashes) == 1:
        return dataframe_parent_hashes[0]

    if isinstance(truth_record, dict):
        truth_parent_hash = truth_record.get(truth_parent_key)
        if truth_parent_hash is not None and str(truth_parent_hash).strip() != "":
            return str(truth_parent_hash)

    raise ValueError(
        "Could not resolve a single parent Gold truth hash from the current Gold 03c inputs."
    )

#### Resolve the parent truth hash for Gold lineage

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [19]:
# =========================================================
# Resolve Stage 3 improved parent Gold truth hash
# =========================================================
# Use globals().get(...) instead of direct notebook variable references.
# This avoids Pylance "not defined" warnings while still behaving correctly
# when the notebook is run top-to-bottom.

gold_preprocessed_scaled_dataframe_object = globals().get(
    "gold_preprocessed_scaled_dataframe"
)
gold_truth_object = globals().get("gold_truth")

if (
    gold_preprocessed_scaled_dataframe_object is not None
    and not isinstance(gold_preprocessed_scaled_dataframe_object, pd.DataFrame)
):
    raise TypeError(
        "gold_preprocessed_scaled_dataframe exists, but it is not a pandas DataFrame."
    )

if gold_truth_object is not None and not isinstance(gold_truth_object, dict):
    raise TypeError("gold_truth exists, but it is not a dictionary.")

STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH = resolve_single_parent_gold_truth_hash(
    gold_preprocessed_scaled_dataframe_object,
    truth_record=gold_truth_object,
)

print("Resolved Stage 3 improved parent Gold truth hash:")
print(STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH)


# =========================================================
# Patch Stage 3 improved lineage fields safely
# =========================================================

cascade_output_object = globals().get("cascade_output")

if isinstance(cascade_output_object, dict):
    cascade_output_summary = cascade_output_object.get("summary")

    if isinstance(cascade_output_summary, dict):
        cascade_output_summary["parent_gold_truth_hash"] = (
            STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH
        )


cascade_summary_object = globals().get("cascade_summary")

if isinstance(cascade_summary_object, dict):
    cascade_summary_object["parent_gold_truth_hash"] = (
        STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH
    )


if isinstance(gold_truth_object, dict):
    runtime_facts = gold_truth_object.setdefault("runtime_facts", {})

    if isinstance(runtime_facts, dict):
        runtime_facts["resolved_parent_gold_truth_hash_for_stage3_improved"] = (
            STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH
        )

Resolved Stage 3 improved parent Gold truth hash:
6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3


----

In [20]:
gold_preprocessed_scaled_dataframe = ensure_stable_row_id(
    gold_preprocessed_scaled_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="validate_cascade_row_tracking",
    message="Validated stable row identity on Gold cascade modeling input dataframe.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_preprocessed_scaled_dataframe)),
        "row_id_unique": bool(gold_preprocessed_scaled_dataframe["meta__row_id"].is_unique),
    },
    logger=logger,
)

2026-06-16 20:43:59,145 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:43:59.145669+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_cascade_row_tracking', 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.', 'why': None, 'consequence': None, 'data': {'row_id_column': 'meta__row_id', 'row_count': 220320, 'row_id_unique': True}}


{'ts_utc': '2026-06-16T20:43:59.145669+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'validate_cascade_row_tracking',
 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.',
 'why': None,
 'consequence': None,
 'data': {'row_id_column': 'meta__row_id',
  'row_count': 220320,
  'row_id_unique': True}}

---

## Rebuild the Train and Test Masks

Before fitting or evaluating the cascade, I recover the train/test split that was already stamped during Gold preprocessing.

The key point here is that this notebook is **not** creating a new split. It is reusing the split created upstream so the baseline notebook, the default cascade notebook, the tuned cascade notebook, and the comparison notebook all work from the same row partition.

In [21]:
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError(
        "meta__is_train_flag missing from gold_preprocessed_scaled_dataframe."
    )

train_mask: pd.Series = (
    gold_preprocessed_scaled_dataframe["meta__is_train_flag"]
    .fillna(False)
    .astype(bool)
)

test_mask: pd.Series = (~train_mask).astype(bool)

train_mask_array: np.ndarray = train_mask.to_numpy(dtype=bool)
test_mask_array: np.ndarray = test_mask.to_numpy(dtype=bool)

test_labels: np.ndarray | None = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

### Ask

Why does the tuned cascade notebook reuse the saved split instead of building a fresh one here?

### Answer

I want the evaluation setup to stay consistent across the Gold modeling notebooks.

If the baseline and cascade notebooks each made their own split, then the results would be harder to compare fairly. By reusing the saved Gold split, I know each modeling approach is being judged against the same training and test boundary.

So this step is mainly about consistency and comparability.

----

## Define the Stage 3 Reference Profile Logic

Before Stage 3 can confirm alerts, I need a reference profile that describes what normal feature behavior looks like.

This helper function builds that profile from the Gold fit dataframe by summarizing each selected feature using values like:
- median
- mean
- standard deviation
- lower bound
- upper bound

The main idea is that Stage 3 should not just look for model flags. It should also compare suspicious rows against a stored picture of normal behavior.

In [22]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

## Build the Stage 3 Reference Profile

Here I create the actual reference profile used by Stage 3 confirmation logic.

The feature set for this profile combines:
- the broader Stage 1 modeling features
- the Stage 3 primary rule sensors
- the Stage 3 secondary rule sensors

That gives Stage 3 a normal-behavior reference it can use when checking for profile breaches and rule evidence.

In [23]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    gold_fit_dataframe,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built fit-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(gold_fit_dataframe)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

display(reference_profile.head(10))

2026-06-16 20:44:00,811 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:00.811387+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'build_reference_profile', 'message': 'Built fit-period reference profile for Stage 3 confirmation logic.', 'why': None, 'consequence': None, 'data': {'training_rows': 77395, 'reference_feature_count': 50}}


,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-0.225564,1.983531,-2.540039,1.100022
1,sensor_01,0.0,0.021708,0.841543,-1.347824,1.456523
2,sensor_02,0.0,-0.011922,0.672431,-1.089288,1.017860
3,sensor_03,0.0,0.060602,0.691585,-0.980768,1.249997
4,sensor_04,0.0,-0.232387,1.951311,-1.927685,1.228905
5,sensor_05,0.0,0.015411,0.868819,-1.361052,1.536601
6,sensor_06,0.0,-0.038234,1.160401,-1.434801,1.804388
7,sensor_07,0.0,-0.164475,0.692269,-1.046894,1.156255
8,sensor_08,0.0,0.147561,0.763275,-0.929813,1.333325
9,sensor_09,0.0,-0.556567,3.136236,-2.666912,0.733364


----

## Prepare the Feature Matrices and Evaluation Labels

Now I build the actual feature matrices that the cascade will use.

This includes:
- Stage 1 fit features from the Gold fit dataframe
- Stage 2 fit features from the Gold fit dataframe
- Stage 1 full-dataset scoring features
- Stage 2 full-dataset scoring features
- test labels, when `anomaly_flag` is available

A detail that matters here is that the model fitting happens on the **normal-only Gold fit subset**, while scoring happens across the **full scaled Gold dataset**.

In [24]:
'''

# Deprecated feature-matrix block removed.
# The active typed feature-matrix construction is handled in the following cell.

# =========================================================
# Build model feature matrices
# =========================================================
# Keep these as DataFrames, not NumPy arrays.
# This prevents the sklearn warning:
# "X has feature names, but IsolationForest was fitted without feature names"
#
# Labels can still use .values/.to_numpy().
# The issue is only with the feature matrix passed into IsolationForest.

# Fit features from normal-only fit parquet
stage1_train_fit_features = gold_fit_dataframe.loc[:, stage1_feature_columns].copy()
stage2_train_fit_features = gold_fit_dataframe.loc[:, stage2_feature_columns].copy()

# Score features from the full scaled dataset, all rows
stage1_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage1_feature_columns].copy()
stage2_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage2_feature_columns].copy()


test_labels = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .values
    )
'''

'\n\n# Deprecated feature-matrix block removed.\n# The active typed feature-matrix construction is handled in the following cell.\n\n# =========================================================\n# Build model feature matrices\n# =========================================================\n# Keep these as DataFrames, not NumPy arrays.\n# This prevents the sklearn warning:\n# "X has feature names, but IsolationForest was fitted without feature names"\n#\n# Labels can still use .values/.to_numpy().\n# The issue is only with the feature matrix passed into IsolationForest.\n\n# Fit features from normal-only fit parquet\nstage1_train_fit_features = gold_fit_dataframe.loc[:, stage1_feature_columns].copy()\nstage2_train_fit_features = gold_fit_dataframe.loc[:, stage2_feature_columns].copy()\n\n# Score features from the full scaled dataset, all rows\nstage1_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage1_feature_columns].copy()\nstage2_all_features = gold_preprocessed_scaled_datafr

#### Run validation guardrails

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [25]:
# =========================================================
# Build model feature matrices
# =========================================================

missing_stage1_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_fit_dataframe.columns
]

missing_stage2_features = [
    column_name
    for column_name in stage2_feature_columns
    if column_name not in gold_fit_dataframe.columns
]

missing_stage1_scaled_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

missing_stage2_scaled_features = [
    column_name
    for column_name in stage2_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

if missing_stage1_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_fit_dataframe:\n"
        f"{missing_stage1_features[:25]}"
    )

if missing_stage2_features:
    raise ValueError(
        "Stage 2 feature columns are missing from gold_fit_dataframe:\n"
        f"{missing_stage2_features[:25]}"
    )

if missing_stage1_scaled_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_stage1_scaled_features[:25]}"
    )

if missing_stage2_scaled_features:
    raise ValueError(
        "Stage 2 feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_stage2_scaled_features[:25]}"
    )

stage1_train_fit_features: pd.DataFrame = gold_fit_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

stage2_train_fit_features: pd.DataFrame = gold_fit_dataframe.loc[
    :,
    stage2_feature_columns,
].copy()

stage1_all_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

stage2_all_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    :,
    stage2_feature_columns,
].copy()

test_labels: np.ndarray | None = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

----

### Ask

Why am I fitting on the Gold fit subset but scoring the full scaled dataset?

### Answer

Because those steps are doing different jobs.

The fit subset gives the cascade its reference view of normal behavior. The full scaled dataset is what I want to score afterward so I can see where alerts appear across all rows.

So the overall logic is:
- fit the Stage 1 and Stage 2 models on the saved normal training subset
- score every row in the scaled Gold dataset
- evaluate the final cascade outcome on the held-out test rows

In [26]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: pd.DataFrame | np.ndarray,
) -> np.ndarray:
    """
    Compute anomaly scores from a fitted IsolationForest model.

    Higher returned values mean more anomalous.

    Notes
    -----
    The feature_matrix can be a DataFrame or NumPy array, but for this
    project we prefer DataFrames so sklearn keeps feature-name tracking.
    """
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores

    return anomaly_scores

#### Ask

This cell performs the next transformation in `Ask`. Review the output or logged message before relying on this result downstream.

----

## Run Stage 1: Broad Isolation Forest Screening

This is the first model stage of the cascade.

Stage 1 uses the broader feature set to act as an initial screen. The goal here is to cast a wider net and identify rows that look suspicious enough to move forward in the cascade.

In this step I:
- fit the Stage 1 Isolation Forest on the Gold fit subset
- score the fit subset and the full scaled dataset
- choose the Stage 1 threshold from the training-score distribution
- create the Stage 1 alert flags

This stage is intentionally broad. It is supposed to surface candidate alerts, not be the final decision by itself.

In [27]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage1_model,
        stage1_train_fit_features,
    ),
    "stage1_train_scores",
)

stage1_all_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage1_model,
        stage1_all_features,
    ),
    "stage1_all_scores",
)

stage1_threshold: float = choose_threshold_value(
    stage1_train_scores,
    STAGE1_THRESHOLD_PERCENTILE,
)

stage1_flags: np.ndarray = (
    stage1_all_scores >= stage1_threshold
).astype(int)

stage1_summary: dict[str, Any] = {
    "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags[test_mask_array].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(stage1_summary["alert_count_all_rows"]),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

2026-06-16 20:44:06,920 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:06.920750+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage1', 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 95.0, 'threshold': 0.4886482835351958, 'feature_count': 50, 'alert_count_all_rows': 103054, 'alert_count_test_rows': 43236}}


{'ts_utc': '2026-06-16T20:44:06.920750+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage1',
 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.',
 'why': None,
 'consequence': None,
 'data': {'estimator_count': 200,
  'threshold_percentile': 95.0,
  'threshold': 0.4886482835351958,
  'feature_count': 50,
  'alert_count_all_rows': 103054,
  'alert_count_test_rows': 43236}}

### Ask

What is Stage 1 really supposed to do in the cascade?

### Answer

Stage 1 is the broad screening step.

Its job is to catch rows that look suspicious enough to deserve a second look. I do not expect Stage 1 alone to be highly selective. In fact, it is normal for Stage 1 to raise more alerts than the later stages, because the later stages are there to narrow and confirm those alerts.

So when I review Stage 1, I mainly care that it is acting as a reasonable first filter rather than as the final answer.

----

In [28]:
stage1_input_df = build_stage_scoring_frame(
    dataframe=gold_preprocessed_scaled_dataframe,
    feature_columns=stage1_feature_columns,
    mask=None,
    row_id_column="meta__row_id",
)

stage1_results_df = score_isolation_forest_stage(
    stage_dataframe=stage1_input_df,
    model=stage1_model,
    feature_columns=stage1_feature_columns,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

cascade_results = merge_stage_results_back(
    master_dataframe=gold_preprocessed_scaled_dataframe.copy(),
    stage_results_dataframe=stage1_results_df,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="score_stage1_with_row_tracking",
    message="Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.",
    data={
        "stage1_row_count": int(len(stage1_results_df)),
        "stage1_flag_count": int(stage1_results_df["stage1_flag"].sum()),
    },
    logger=logger,
)

2026-06-16 20:44:17,603 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:17.602936+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage1_with_row_tracking', 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 91594}}


{'ts_utc': '2026-06-16T20:44:17.602936+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage1_with_row_tracking',
 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 91594}}

#### Build review visualization

This cell builds a visual review artifact for interpretation and reporting. The plot is used to explain model behavior; it does not change the modeling inputs.

In [29]:
# =========================================================
# Ensure plot_order_index exists after row-tracked Stage 1 merge
# =========================================================
# This gives every later debug display, lead-time check, and timeline plot
# a stable row-order column to use.

if "plot_order_index" not in cascade_results.columns:
    if "time_index" in cascade_results.columns:
        cascade_results["plot_order_index"] = pd.to_numeric(
            cascade_results["time_index"],
            errors="coerce",
        )

        if cascade_results["plot_order_index"].isna().any():
            cascade_results["plot_order_index"] = np.arange(len(cascade_results))

        print("Created plot_order_index from time_index.")

    elif "meta__row_id" in cascade_results.columns:
        row_id_as_number = pd.to_numeric(
            cascade_results["meta__row_id"],
            errors="coerce",
        )

        if row_id_as_number.notna().all():
            cascade_results["plot_order_index"] = row_id_as_number
            print("Created plot_order_index from numeric meta__row_id.")
        else:
            cascade_results["plot_order_index"] = np.arange(len(cascade_results))
            print("Created plot_order_index from dataframe row order.")

    else:
        cascade_results["plot_order_index"] = np.arange(len(cascade_results))
        print("Created plot_order_index from dataframe row order.")

cascade_results["plot_order_index"] = cascade_results["plot_order_index"].astype(int)

ledger.add(
    kind="step",
    step="ensure_plot_order_index",
    message="Ensured cascade_results contains plot_order_index for debug displays, lead-time checks, and timeline plotting.",
    data={
        "plot_order_index_min": int(cascade_results["plot_order_index"].min()),
        "plot_order_index_max": int(cascade_results["plot_order_index"].max()),
        "cascade_result_rows": int(len(cascade_results)),
    },
    logger=logger,
)

print("plot_order_index ready.")
print(
    {
        "min": int(cascade_results["plot_order_index"].min()),
        "max": int(cascade_results["plot_order_index"].max()),
        "rows": int(len(cascade_results)),
    }
)

2026-06-16 20:44:18,253 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:18.253835+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'ensure_plot_order_index', 'message': 'Ensured cascade_results contains plot_order_index for debug displays, lead-time checks, and timeline plotting.', 'why': None, 'consequence': None, 'data': {'plot_order_index_min': 0, 'plot_order_index_max': 220319, 'cascade_result_rows': 220320}}


Created plot_order_index from time_index.
plot_order_index ready.
{'min': 0, 'max': 220319, 'rows': 220320}


#### Build review visualization

This cell builds a visual review artifact for interpretation and reporting. The plot is used to explain model behavior; it does not change the modeling inputs.

In [30]:
# =========================================================
# Synchronize Stage 1 threshold flags after row-tracked scoring
# =========================================================

if len(stage1_all_scores) != len(cascade_results):
    raise ValueError(
        "stage1_all_scores length does not match cascade_results length. "
        "Stage 1 row-tracked synchronization is unsafe."
    )

cascade_results["stage1_score"] = stage1_all_scores
cascade_results["stage1_threshold"] = float(stage1_threshold)
cascade_results["stage1_threshold_percentile"] = float(STAGE1_THRESHOLD_PERCENTILE)
cascade_results["stage1_flag"] = stage1_flags.astype(int)

ledger.add(
    kind="step",
    step="synchronize_stage1_threshold_flags",
    message="Synchronized Stage 1 row-tracked results with the configured percentile-threshold alert rule.",
    data={
        "stage1_threshold": float(stage1_threshold),
        "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
        "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    },
    logger=logger,
)

print("Stage 1 threshold-synchronized alert count:", int(cascade_results["stage1_flag"].sum()))

2026-06-16 20:44:18,762 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:18.762917+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'synchronize_stage1_threshold_flags', 'message': 'Synchronized Stage 1 row-tracked results with the configured percentile-threshold alert rule.', 'why': None, 'consequence': None, 'data': {'stage1_threshold': 0.4886482835351958, 'stage1_threshold_percentile': 95.0, 'stage1_alert_count_all_rows': 103054, 'stage1_alert_count_test_rows': 43236}}


Stage 1 threshold-synchronized alert count: 103054


---

## Define the Stage 2 Selection Logic

This helper section defines how the tuned Stage 2 model is chosen.

Instead of assuming one fixed Stage 2 setup, this notebook evaluates candidate model and threshold combinations and keeps the best result based on the configured selection score. That score is designed to reward stronger alert quality while still respecting the minimum recall requirement.

So this is the part of the notebook where the tuned cascade starts to differ from the default cascade version.

#### Evaluate Stage 2 threshold candidates

This cell defines helper logic used by the surrounding `----` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [31]:
def evaluate_stage2_model_with_thresholds(
    *,
    model: IsolationForest,
    model_params: dict,
    stage2_train_fit_features: pd.DataFrame | np.ndarray,
    stage2_all_features: pd.DataFrame | np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    threshold_percentiles: list[float],
    min_recall: float,
) -> dict[str, Any]:
    model.fit(stage2_train_fit_features)

    stage2_train_scores: np.ndarray = as_float_array(
        compute_anomaly_scores_isolation_forest(
            model,
            stage2_train_fit_features,
        ),
        "stage2_train_scores",
    )

    stage2_all_scores: np.ndarray = as_float_array(
        compute_anomaly_scores_isolation_forest(
            model,
            stage2_all_features,
        ),
        "stage2_all_scores",
    )

    test_mask_array_local: np.ndarray = test_mask.to_numpy(dtype=bool)

    best_result: dict[str, Any] | None = None

    for threshold_percentile in threshold_percentiles:
        stage2_threshold: float = choose_threshold_value(
            stage2_train_scores,
            float(threshold_percentile),
        )

        stage2_raw_flags: np.ndarray = (
            stage2_all_scores >= stage2_threshold
        ).astype(int)

        stage2_flags: np.ndarray = (
            (stage1_flags == 1)
            & (stage2_raw_flags == 1)
        ).astype(int)

        if test_labels is not None:
            test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)
            stage2_test_flags: np.ndarray = stage2_flags[test_mask_array_local]

            precision, recall, f1, _ = precision_recall_fscore_support(
                test_labels_array,
                stage2_test_flags,
                average="binary",
                zero_division=0,
            )

            tn, fp, fn, tp = confusion_matrix(
                test_labels_array,
                stage2_test_flags,
                labels=[0, 1],
            ).ravel()

            stage2_confirmed_count_test_rows = int(stage2_test_flags.sum())
            test_row_count = int(len(stage2_test_flags))

            alert_rate = (
                float(stage2_confirmed_count_test_rows)
                / float(max(test_row_count, 1))
            )

            if float(recall) < float(min_recall):
                selection_score = -1000.0 + float(recall)
            else:
                selection_score = (
                    (3.0 * float(f1))
                    + (1.0 * float(precision))
                    - (1.0 * float(alert_rate))
                )
        else:
            precision = None
            recall = None
            f1 = None
            tn = None
            fp = None
            fn = None
            tp = None
            stage2_confirmed_count_test_rows = int(
                stage2_flags[test_mask_array_local].sum()
            )
            test_row_count = int(test_mask_array_local.sum())
            alert_rate = (
                float(stage2_confirmed_count_test_rows)
                / float(max(test_row_count, 1))
            )
            selection_score = -float(stage2_confirmed_count_test_rows)

        candidate_result: dict[str, Any] = {
            "model_params": model_params,
            "selected_threshold_percentile": float(threshold_percentile),
            "stage2_threshold": float(stage2_threshold),
            "stage2_train_scores": stage2_train_scores,
            "stage2_all_scores": stage2_all_scores,
            "stage2_raw_flags": stage2_raw_flags,
            "stage2_flags": stage2_flags,
            "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
            "raw_alert_count_test_rows": int(stage2_raw_flags[test_mask_array_local].sum()),
            "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
            "stage2_confirmed_count_test_rows": int(stage2_confirmed_count_test_rows),
            "alert_rate_test_rows": float(alert_rate),
            "precision": None if precision is None else float(precision),
            "recall": None if recall is None else float(recall),
            "f1": None if f1 is None else float(f1),
            "selection_score": float(selection_score),
            "tn": None if tn is None else int(tn),
            "fp": None if fp is None else int(fp),
            "fn": None if fn is None else int(fn),
            "tp": None if tp is None else int(tp),
        }

        if (
            best_result is None
            or float(candidate_result["selection_score"]) > float(best_result["selection_score"])
        ):
            best_result = candidate_result

    if best_result is None:
        raise ValueError("Stage 2 threshold evaluation did not produce a valid result.")

    return best_result

### Ask

What is the actual tuning idea behind Stage 2 here?

### Answer

The point is not to blindly search for the biggest score. The point is to choose a narrower Stage 2 setup that still behaves well on the held-out evaluation logic.

In this notebook, each candidate setup is judged using:
- the selected threshold percentile
- the resulting confirmed alert counts
- precision, recall, and F1 on the test rows when labels are available
- a selection score that gives the most weight to precision, while still keeping recall and F1 involved
- a minimum recall rule so overly strict candidates do not automatically win

So Stage 2 tuning here is really a controlled search for a better confirmation layer, not a generic hyperparameter hunt.

----

In [32]:
# =========================================================
# Load prior 03B tuned Stage 2 result
# =========================================================
# 03C always loads the prior 03B best Stage 2 settings.
#
# Stage 2 selection source options:
#   "previous_best"     -> reuse 03B's saved best params + threshold percentile
#   "configured_search" -> run 03C's configured STAGE2_SELECTION_MODE
#   "auto"              -> use previous_best if available, otherwise configured_search

STAGE2_SELECTION_SOURCE = "previous_best"
# STAGE2_SELECTION_SOURCE = "configured_search"
# STAGE2_SELECTION_SOURCE = "auto"

previous_03b_stage2_thresholds: dict[str, Any] = require_mapping(
    load_json(CASCADE_TUNED_THRESHOLDS_PATH),
    "previous_03b_stage2_thresholds",
)

previous_03b_stage2_selected_threshold_percentile = float(
    previous_03b_stage2_thresholds.get(
        "stage2_selected_threshold_percentile",
        STAGE2_FIXED_THRESHOLD_PERCENTILE,
    )
)

raw_previous_03b_stage2_best_params = previous_03b_stage2_thresholds.get(
    "stage2_best_params",
    {},
)

if not isinstance(raw_previous_03b_stage2_best_params, dict):
    raise TypeError(
        "Expected previous 03B stage2_best_params to be a dictionary. "
        f"Got {type(raw_previous_03b_stage2_best_params).__name__}: "
        f"{raw_previous_03b_stage2_best_params!r}"
    )

previous_03b_stage2_best_params: dict[str, Any] = {
    str(key): value
    for key, value in raw_previous_03b_stage2_best_params.items()
}

previous_03b_stage2_available = bool(previous_03b_stage2_best_params)

ledger.add(
    kind="step",
    step="load_prior_03b_stage2_reference",
    message="Loaded prior 03B Stage 2 selected settings for optional reuse in 03C.",
    data={
        "stage2_selection_source": STAGE2_SELECTION_SOURCE,
        "previous_03b_stage2_available": previous_03b_stage2_available,
        "previous_03b_stage2_selected_threshold_percentile": previous_03b_stage2_selected_threshold_percentile,
        "previous_03b_stage2_best_params": previous_03b_stage2_best_params,
        "current_03c_stage2_selection_mode": STAGE2_SELECTION_MODE,
    },
    logger=logger,
)

print("Prior 03B Stage 2 settings loaded.")
print(
    {
        "stage2_selection_source": STAGE2_SELECTION_SOURCE,
        "previous_03b_available": previous_03b_stage2_available,
        "previous_03b_threshold_percentile": previous_03b_stage2_selected_threshold_percentile,
        "previous_03b_best_params": previous_03b_stage2_best_params,
        "current_03c_stage2_selection_mode": STAGE2_SELECTION_MODE,
    }
)

2026-06-16 20:44:19,662 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/cascade_tuned/thresholds/pump__gold__cascade_tuned_thresholds.json
2026-06-16 20:44:19,679 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:19.679618+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'load_prior_03b_stage2_reference', 'message': 'Loaded prior 03B Stage 2 selected settings for optional reuse in 03C.', 'why': None, 'consequence': None, 'data': {'stage2_selection_source': 'previous_best', 'previous_03b_stage2_available': True, 'previous_03b_stage2_selected_threshold_percentile': 99.5, 'previous_03b_stage2_best_params': {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}, 'current_03c_stage2_selection_mode': 'parameter_search'}}


Prior 03B Stage 2 settings loaded.
{'stage2_selection_source': 'previous_best', 'previous_03b_available': True, 'previous_03b_threshold_percentile': 99.5, 'previous_03b_best_params': {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}, 'current_03c_stage2_selection_mode': 'parameter_search'}


In [33]:
def evaluate_stage2_model_with_thresholds(
    *,
    model: IsolationForest,
    model_params: dict,
    stage2_train_fit_features: pd.DataFrame | np.ndarray,
    stage2_all_features: pd.DataFrame | np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    threshold_percentiles: list[float],
    min_recall: float,
) -> dict[str, Any]:
    model.fit(stage2_train_fit_features)

    stage2_train_scores: np.ndarray = as_float_array(
        compute_anomaly_scores_isolation_forest(
            model,
            stage2_train_fit_features,
        ),
        "stage2_train_scores",
    )

    stage2_all_scores: np.ndarray = as_float_array(
        compute_anomaly_scores_isolation_forest(
            model,
            stage2_all_features,
        ),
        "stage2_all_scores",
    )

    test_mask_array_local: np.ndarray = test_mask.to_numpy(dtype=bool)

    best_result: dict[str, Any] | None = None

    for threshold_percentile in threshold_percentiles:
        stage2_threshold: float = choose_threshold_value(
            stage2_train_scores,
            float(threshold_percentile),
        )

        stage2_raw_flags: np.ndarray = (
            stage2_all_scores >= stage2_threshold
        ).astype(int)

        stage2_flags: np.ndarray = (
            (stage1_flags == 1)
            & (stage2_raw_flags == 1)
        ).astype(int)

        if test_labels is not None:
            test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)
            stage2_test_flags: np.ndarray = stage2_flags[test_mask_array_local]

            precision, recall, f1, _ = precision_recall_fscore_support(
                test_labels_array,
                stage2_test_flags,
                average="binary",
                zero_division=0,
            )

            tn, fp, fn, tp = confusion_matrix(
                test_labels_array,
                stage2_test_flags,
                labels=[0, 1],
            ).ravel()

            stage2_confirmed_count_test_rows = int(stage2_test_flags.sum())
            test_row_count = int(len(stage2_test_flags))

            alert_rate = (
                float(stage2_confirmed_count_test_rows)
                / float(max(test_row_count, 1))
            )

            if float(recall) < float(min_recall):
                selection_score = -1000.0 + float(recall)
            else:
                selection_score = (
                    (3.0 * float(f1))
                    + (1.0 * float(precision))
                    - (1.0 * float(alert_rate))
                )
        else:
            precision = None
            recall = None
            f1 = None
            tn = None
            fp = None
            fn = None
            tp = None
            stage2_confirmed_count_test_rows = int(
                stage2_flags[test_mask_array_local].sum()
            )
            test_row_count = int(test_mask_array_local.sum())
            alert_rate = (
                float(stage2_confirmed_count_test_rows)
                / float(max(test_row_count, 1))
            )
            selection_score = -float(stage2_confirmed_count_test_rows)

        candidate_result: dict[str, Any] = {
            "model_params": model_params,
            "selected_threshold_percentile": float(threshold_percentile),
            "stage2_threshold": float(stage2_threshold),
            "stage2_train_scores": stage2_train_scores,
            "stage2_all_scores": stage2_all_scores,
            "stage2_raw_flags": stage2_raw_flags,
            "stage2_flags": stage2_flags,
            "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
            "raw_alert_count_test_rows": int(stage2_raw_flags[test_mask_array_local].sum()),
            "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
            "stage2_confirmed_count_test_rows": int(stage2_confirmed_count_test_rows),
            "alert_rate_test_rows": float(alert_rate),
            "precision": None if precision is None else float(precision),
            "recall": None if recall is None else float(recall),
            "f1": None if f1 is None else float(f1),
            "selection_score": float(selection_score),
            "tn": None if tn is None else int(tn),
            "fp": None if fp is None else int(fp),
            "fn": None if fn is None else int(fn),
            "tp": None if tp is None else int(tp),
        }

        if (
            best_result is None
            or float(candidate_result["selection_score"]) > float(best_result["selection_score"])
        ):
            best_result = candidate_result

    if best_result is None:
        raise ValueError("Stage 2 threshold evaluation did not produce a valid result.")

    return best_result


def run_stage2_selection(
    *,
    selection_mode: str,
    fixed_params: dict,
    fixed_threshold_percentile: float,
    threshold_grid: list[float],
    param_grid: dict,
    stage2_train_fit_features: pd.DataFrame | np.ndarray,
    stage2_all_features: pd.DataFrame | np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    random_seed: int,
    min_recall: float,
) -> tuple[IsolationForest, dict[str, Any], pd.DataFrame]:
    selection_mode_clean = str(selection_mode).strip().lower()

    search_rows: list[dict[str, Any]] = []
    best_result: dict[str, Any] | None = None
    best_model: IsolationForest | None = None

    if selection_mode_clean == "fixed":
        model_candidates = [fixed_params.copy()]
        threshold_candidates = [float(fixed_threshold_percentile)]

    elif selection_mode_clean == "threshold_grid":
        model_candidates = [fixed_params.copy()]
        threshold_candidates = [float(value) for value in threshold_grid]

    elif selection_mode_clean == "parameter_search":
        model_candidates = [dict(params) for params in ParameterGrid(param_grid)]
        threshold_candidates = [float(value) for value in threshold_grid]

    else:
        raise ValueError(
            f"Unsupported STAGE2_SELECTION_MODE: {selection_mode}. "
            "Use 'fixed', 'threshold_grid', or 'parameter_search'."
        )

    for model_params in model_candidates:
        candidate_model = IsolationForest(
            random_state=random_seed,
            n_jobs=-1,
            **model_params,
        )

        candidate_result = evaluate_stage2_model_with_thresholds(
            model=candidate_model,
            model_params=model_params,
            stage2_train_fit_features=stage2_train_fit_features,
            stage2_all_features=stage2_all_features,
            stage1_flags=stage1_flags,
            test_mask=test_mask,
            test_labels=test_labels,
            threshold_percentiles=threshold_candidates,
            min_recall=min_recall,
        )

        search_rows.append(
            {
                **candidate_result["model_params"],
                "selected_threshold_percentile": candidate_result["selected_threshold_percentile"],
                "stage2_threshold": candidate_result["stage2_threshold"],
                "raw_alert_count_all_rows": candidate_result["raw_alert_count_all_rows"],
                "raw_alert_count_test_rows": candidate_result["raw_alert_count_test_rows"],
                "stage2_confirmed_count_all_rows": candidate_result["stage2_confirmed_count_all_rows"],
                "stage2_confirmed_count_test_rows": candidate_result["stage2_confirmed_count_test_rows"],
                "alert_rate_test_rows": candidate_result.get("alert_rate_test_rows"),
                "precision": candidate_result["precision"],
                "recall": candidate_result["recall"],
                "f1": candidate_result["f1"],
                "selection_score": candidate_result["selection_score"],
                "tn": candidate_result["tn"],
                "fp": candidate_result["fp"],
                "fn": candidate_result["fn"],
                "tp": candidate_result["tp"],
            }
        )

        if (
            best_result is None
            or float(candidate_result["selection_score"]) > float(best_result["selection_score"])
        ):
            best_result = candidate_result
            best_model = candidate_model

    if best_result is None or best_model is None:
        raise ValueError("Stage 2 selection did not produce a best model.")

    search_results = (
        pd.DataFrame(search_rows)
        .sort_values(
            by=["selection_score"],
            ascending=[False],
        )
        .reset_index(drop=True)
    )

    return best_model, best_result, search_results


def run_stage2_previous_best(
    *,
    previous_best_params: dict[str, Any],
    previous_threshold_percentile: float,
    stage2_train_fit_features: pd.DataFrame | np.ndarray,
    stage2_all_features: pd.DataFrame | np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    random_seed: int,
    min_recall: float,
) -> tuple[IsolationForest, dict[str, Any], pd.DataFrame]:
    """
    Reuse the previously selected 03B Stage 2 configuration.

    This refits a Stage 2 model on the current 03C Gold fit data using the
    previous best hyperparameters, then applies the previous selected threshold
    percentile to the current training score distribution.
    """
    if not previous_best_params:
        raise ValueError(
            "Cannot use previous_best Stage 2 selection because "
            "previous_best_params is empty."
        )

    previous_best_params_clean: dict[str, Any] = {
        str(key): value
        for key, value in previous_best_params.items()
    }

    stage2_model = IsolationForest(
        random_state=random_seed,
        n_jobs=-1,
        **previous_best_params_clean,
    )

    best_result = evaluate_stage2_model_with_thresholds(
        model=stage2_model,
        model_params=previous_best_params_clean,
        stage2_train_fit_features=stage2_train_fit_features,
        stage2_all_features=stage2_all_features,
        stage1_flags=stage1_flags,
        test_mask=test_mask,
        test_labels=test_labels,
        threshold_percentiles=[float(previous_threshold_percentile)],
        min_recall=min_recall,
    )

    best_result["selection_source"] = "previous_best"
    best_result["selection_mode"] = "previous_best"
    best_result["reused_previous_03b_best"] = True

    stage2_search_results = pd.DataFrame(
        [
            {
                **previous_best_params_clean,
                "selection_source": "previous_best",
                "selection_mode": "previous_best",
                "selected_threshold_percentile": best_result["selected_threshold_percentile"],
                "stage2_threshold": best_result["stage2_threshold"],
                "raw_alert_count_all_rows": best_result["raw_alert_count_all_rows"],
                "raw_alert_count_test_rows": best_result["raw_alert_count_test_rows"],
                "stage2_confirmed_count_all_rows": best_result["stage2_confirmed_count_all_rows"],
                "stage2_confirmed_count_test_rows": best_result["stage2_confirmed_count_test_rows"],
                "alert_rate_test_rows": best_result.get("alert_rate_test_rows"),
                "precision": best_result.get("precision"),
                "recall": best_result.get("recall"),
                "f1": best_result.get("f1"),
                "selection_score": best_result.get("selection_score"),
                "tn": best_result.get("tn"),
                "fp": best_result.get("fp"),
                "fn": best_result.get("fn"),
                "tp": best_result.get("tp"),
            }
        ]
    )

    return stage2_model, best_result, stage2_search_results


def run_stage2_selection_decision(
    *,
    selection_source: str,
    previous_best_available: bool,
    previous_best_params: dict[str, Any],
    previous_threshold_percentile: float,
    selection_mode: str,
    fixed_params: dict,
    fixed_threshold_percentile: float,
    threshold_grid: list[float],
    param_grid: dict,
    stage2_train_fit_features: pd.DataFrame | np.ndarray,
    stage2_all_features: pd.DataFrame | np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    random_seed: int,
    min_recall: float,
) -> tuple[IsolationForest, dict[str, Any], pd.DataFrame]:
    """
    Decide whether 03C should reuse 03B's previous best Stage 2 settings
    or run a fresh configured Stage 2 search.
    """
    selection_source_clean = str(selection_source).strip().lower()

    if selection_source_clean == "auto":
        if previous_best_available:
            selection_source_clean = "previous_best"
        else:
            selection_source_clean = "configured_search"

    if selection_source_clean in {"previous_best", "previous", "reuse_previous", "reuse_03b"}:
        return run_stage2_previous_best(
            previous_best_params=previous_best_params,
            previous_threshold_percentile=previous_threshold_percentile,
            stage2_train_fit_features=stage2_train_fit_features,
            stage2_all_features=stage2_all_features,
            stage1_flags=stage1_flags,
            test_mask=test_mask,
            test_labels=test_labels,
            random_seed=random_seed,
            min_recall=min_recall,
        )

    if selection_source_clean in {"configured_search", "search", "grid_search", "run_search"}:
        stage2_model, best_result, stage2_search_results = run_stage2_selection(
            selection_mode=selection_mode,
            fixed_params=fixed_params,
            fixed_threshold_percentile=fixed_threshold_percentile,
            threshold_grid=threshold_grid,
            param_grid=param_grid,
            stage2_train_fit_features=stage2_train_fit_features,
            stage2_all_features=stage2_all_features,
            stage1_flags=stage1_flags,
            test_mask=test_mask,
            test_labels=test_labels,
            random_seed=random_seed,
            min_recall=min_recall,
        )

        best_result["selection_source"] = "configured_search"
        best_result["selection_mode"] = selection_mode
        best_result["reused_previous_03b_best"] = False

        return stage2_model, best_result, stage2_search_results

    raise ValueError(
        "Unsupported STAGE2_SELECTION_SOURCE. "
        "Use 'previous_best', 'configured_search', or 'auto'. "
        f"Got: {selection_source!r}"
    )

----

## Run Stage 2 Selection and Keep the Best Narrow Model

This is the tuned Stage 2 step of the cascade.

Here I run the configured Stage 2 selection process, which may operate in one of several modes:
- fixed
- threshold grid
- parameter search

For the tuned cascade, the goal is to search the candidate space and keep the narrow Isolation Forest setup that gives the best selection result under the notebook's rules.

After that, I keep:
- the best Stage 2 model
- the selected threshold percentile
- the chosen Stage 2 threshold
- the best parameter set
- the full search results table for review

In [34]:
stage2_model, best_stage2_result, stage2_search_results = run_stage2_selection_decision(
    selection_source=STAGE2_SELECTION_SOURCE,
    previous_best_available=previous_03b_stage2_available,
    previous_best_params=previous_03b_stage2_best_params,
    previous_threshold_percentile=previous_03b_stage2_selected_threshold_percentile,
    selection_mode=STAGE2_SELECTION_MODE,
    fixed_params=STAGE2_FIXED_PARAMS,
    fixed_threshold_percentile=STAGE2_FIXED_THRESHOLD_PERCENTILE,
    threshold_grid=STAGE2_THRESHOLD_GRID,
    param_grid=STAGE2_PARAM_GRID,
    stage2_train_fit_features=stage2_train_fit_features,
    stage2_all_features=stage2_all_features,
    stage1_flags=stage1_flags,
    test_mask=test_mask,
    test_labels=test_labels,
    random_seed=STAGE2_RANDOM_STATE,
    min_recall=STAGE2_MIN_RECALL,
)

stage2_train_scores: np.ndarray = as_float_array(
    best_stage2_result["stage2_train_scores"],
    "stage2_train_scores",
)

stage2_all_scores: np.ndarray = as_float_array(
    best_stage2_result["stage2_all_scores"],
    "stage2_all_scores",
)

stage2_threshold = float(best_stage2_result["stage2_threshold"])

stage2_raw_flags: np.ndarray = as_int_array(
    best_stage2_result["stage2_raw_flags"],
    "stage2_raw_flags",
)

stage2_flags: np.ndarray = as_int_array(
    best_stage2_result["stage2_flags"],
    "stage2_flags",
)

stage2_selected_threshold_percentile = float(
    best_stage2_result["selected_threshold_percentile"]
)

raw_stage2_best_params = best_stage2_result["model_params"]

if not isinstance(raw_stage2_best_params, dict):
    raise TypeError(
        "Expected best_stage2_result['model_params'] to be a dictionary. "
        f"Got {type(raw_stage2_best_params).__name__}: {raw_stage2_best_params!r}"
    )

stage2_best_params: dict[str, Any] = {
    str(key): value
    for key, value in raw_stage2_best_params.items()
}

stage2_selection_source_used = str(
    best_stage2_result.get("selection_source", STAGE2_SELECTION_SOURCE)
)

stage2_selection_mode_used = str(
    best_stage2_result.get("selection_mode", STAGE2_SELECTION_MODE)
)

stage2_summary: dict[str, Any] = {
    "selection_source": stage2_selection_source_used,
    "selection_mode": stage2_selection_mode_used,
    "reused_previous_03b_best": bool(
        best_stage2_result.get("reused_previous_03b_best", False)
    ),
    "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "threshold": float(stage2_threshold),
    "best_params": stage2_best_params,
    "raw_alert_count_all_rows": int(best_stage2_result["raw_alert_count_all_rows"]),
    "raw_alert_count_test_rows": int(best_stage2_result["raw_alert_count_test_rows"]),
    "stage2_confirmed_count_all_rows": int(best_stage2_result["stage2_confirmed_count_all_rows"]),
    "stage2_confirmed_count_test_rows": int(best_stage2_result["stage2_confirmed_count_test_rows"]),
    "precision": None
    if best_stage2_result.get("precision") is None
    else float(best_stage2_result["precision"]),
    "recall": None
    if best_stage2_result.get("recall") is None
    else float(best_stage2_result["recall"]),
    "f1": None
    if best_stage2_result.get("f1") is None
    else float(best_stage2_result["f1"]),
    "selection_score": float(best_stage2_result["selection_score"]),
    "candidate_count": int(len(stage2_search_results)),
}

ledger.add(
    kind="step",
    step="run_cascade_stage2_selection",
    message="Selected Stage 2 configuration using either prior 03B best settings or a configured 03C search.",
    data=stage2_summary,
    logger=logger,
)

display(stage2_summary)
stage2_search_results.head(10)

2026-06-16 20:44:22,417 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:22.417217+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage2_selection', 'message': 'Selected Stage 2 configuration using either prior 03B best settings or a configured 03C search.', 'why': None, 'consequence': None, 'data': {'selection_source': 'previous_best', 'selection_mode': 'previous_best', 'reused_previous_03b_best': True, 'selected_threshold_percentile': 99.5, 'threshold': 0.584003211617404, 'best_params': {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}, 'raw_alert_count_all_rows': 57827, 'raw_alert_count_test_rows': 18250, 'stage2_confirmed_count_all_rows': 55684, 'stage2_confirmed_count_test_rows': 16108, 'precision': 0.005028557238639185, 'recall': 0.6864406779661016, 'f1': 0.009983976334278319, 'selection_score': -0.15703515347291033, 'c

{'selection_source': 'previous_best',
 'selection_mode': 'previous_best',
 'reused_previous_03b_best': True,
 'selected_threshold_percentile': 99.5,
 'threshold': 0.584003211617404,
 'best_params': {'bootstrap': False,
  'contamination': 'auto',
  'max_features': 1.0,
  'max_samples': 'auto',
  'n_estimators': 100},
 'raw_alert_count_all_rows': 57827,
 'raw_alert_count_test_rows': 18250,
 'stage2_confirmed_count_all_rows': 55684,
 'stage2_confirmed_count_test_rows': 16108,
 'precision': 0.005028557238639185,
 'recall': 0.6864406779661016,
 'f1': 0.009983976334278319,
 'selection_score': -0.15703515347291033,
 'candidate_count': 1}

,bootstrap,contamination,max_features,max_samples,n_estimators,selection_source,selection_mode,selected_threshold_percentile,stage2_threshold,raw_alert_count_all_rows,raw_alert_count_test_rows,stage2_confirmed_count_all_rows,stage2_confirmed_count_test_rows,alert_rate_test_rows,precision,recall,f1,selection_score,tn,fp,fn,tp
0,False,auto,1.0,auto,100,previous_best,previous_best,99.5,0.584003,57827,18250,55684,16108,0.192016,0.005029,0.686441,0.009984,-0.157035,67744,16027,37,81


### Ask

What should I pay attention to when I look at the Stage 2 search results?

### Answer

The main things I care about are:
- what `selection_mode` was used
- how many candidate combinations were evaluated
- which parameter set won
- which threshold percentile was selected
- how many raw Stage 2 alerts were produced
- how many confirmed Stage 2 alerts survived after the Stage 1 gate
- whether the winning candidate looks balanced rather than simply aggressive or overly quiet

This is important because the tuned cascade should not just be "different" from the default one. It should have a clear reason for why its Stage 2 branch was selected.

---

In [35]:
stage2_candidate_mask = (
    cascade_results["stage1_flag"]
    .fillna(0)
    .astype(int)
    == 1
)

stage2_candidate_mask_array: np.ndarray = stage2_candidate_mask.to_numpy(dtype=bool)

stage2_input_df = build_stage_scoring_frame(
    dataframe=cascade_results,
    feature_columns=stage2_feature_columns,
    mask=stage2_candidate_mask,
    row_id_column="meta__row_id",
)

stage2_results_df = score_isolation_forest_stage(
    stage_dataframe=stage2_input_df,
    model=stage2_model,
    feature_columns=stage2_feature_columns,
    stage_name="stage2",
    row_id_column="meta__row_id",
)

# Rename raw helper outputs so they do not overwrite your threshold-based final stage2 flag logic.
stage2_results_df = stage2_results_df.rename(
    columns={
        "stage2_score": "stage2_model_score",
        "stage2_decision": "stage2_model_decision",
        "stage2_pred": "stage2_model_pred",
        "stage2_flag": "stage2_model_flag",
    }
)

cascade_results = cascade_results.merge(
    stage2_results_df[
        [
            "meta__row_id",
            "stage2_model_score",
            "stage2_model_decision",
            "stage2_model_pred",
            "stage2_model_flag",
        ]
    ],
    on="meta__row_id",
    how="left",
)

cascade_results["stage2_score"] = np.nan
cascade_results.loc[stage2_candidate_mask, "stage2_score"] = stage2_all_scores[
    stage2_candidate_mask_array
]

cascade_results["stage2_raw_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_raw_flag"] = stage2_raw_flags[
    stage2_candidate_mask_array
]

cascade_results["stage2_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_flag"] = stage2_flags[
    stage2_candidate_mask_array
]

cascade_results["stage2_raw_flag"] = (
    cascade_results["stage2_raw_flag"]
    .fillna(0)
    .astype(int)
)

cascade_results["stage2_flag"] = (
    cascade_results["stage2_flag"]
    .fillna(0)
    .astype(int)
)

ledger.add(
    kind="step",
    step="score_stage2_with_row_tracking",
    message="Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.",
    data={
        "stage2_candidate_count": int(stage2_candidate_mask.sum()),
        "stage2_scored_row_count": int(len(stage2_results_df)),
        "stage2_model_flag_count": int(stage2_results_df["stage2_model_flag"].sum()),
        "stage2_final_flag_count": int(cascade_results["stage2_flag"].sum()),
    },
    logger=logger,
)

2026-06-16 20:44:26,005 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:44:26.005562+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage2_with_row_tracking', 'message': 'Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage2_candidate_count': 103054, 'stage2_scored_row_count': 103054, 'stage2_model_flag_count': 99176, 'stage2_final_flag_count': 55684}}


{'ts_utc': '2026-06-16T20:44:26.005562+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage2_with_row_tracking',
 'message': 'Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage2_candidate_count': 103054,
  'stage2_scored_row_count': 103054,
  'stage2_model_flag_count': 99176,
  'stage2_final_flag_count': 55684}}

----

#### Quick Verifications Cell 

In [36]:
stage2_search_results_object = globals().get("stage2_search_results")

print("STAGE2_SELECTION_MODE:", STAGE2_SELECTION_MODE)
print("stage2_best_params:", stage2_best_params)

if stage2_search_results_object is None:
    print("search candidate count: missing")
else:
    print("search candidate count:", len(stage2_search_results_object))

    if len(stage2_search_results_object) > 0:
        display(pd.DataFrame(stage2_search_results_object).head())

STAGE2_SELECTION_MODE: parameter_search
stage2_best_params: {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}
search candidate count: 1


,bootstrap,contamination,max_features,max_samples,n_estimators,selection_source,selection_mode,selected_threshold_percentile,stage2_threshold,raw_alert_count_all_rows,raw_alert_count_test_rows,stage2_confirmed_count_all_rows,stage2_confirmed_count_test_rows,alert_rate_test_rows,precision,recall,f1,selection_score,tn,fp,fn,tp
0,False,auto,1.0,auto,100,previous_best,previous_best,99.5,0.584003,57827,18250,55684,16108,0.192016,0.005029,0.686441,0.009984,-0.157035,67744,16027,37,81


----

## Continue the Cascade Results Dataframe from Row-Tracked Stage Outputs

At this point the working `cascade_results` dataframe already contains the Stage 1 row-level outputs merged by stable row identity, and Stage 2 row-level outputs are now merged back in for the Stage 1 candidate rows.

So I do not rebuild the results dataframe from scratch here. Instead, I continue using the row-tracked `cascade_results` dataframe as the single source of truth for Stage 3 rule logic.

----

## Validate That the Stage 3 Rule Sensors Exist

Before I run Stage 3 confirmation logic, I want to verify that the required primary and secondary rule sensors are actually present in the scored dataframe.

This is a quick trust check. If those sensors are missing, then the Stage 3 confirmation logic would be working from an incomplete rule set, which would weaken the final cascade decision.

In [37]:
# --- Stage 3 sanity check: ensure rule sensors exist in scored dataframe
missing_primary = [column for column in stage3_primary_rule_sensors if column not in cascade_results.columns]
missing_secondary = [column for column in stage3_secondary_rule_sensors if column not in cascade_results.columns]

logger.info("Stage3 missing sensors: primary=%d secondary=%d", len(missing_primary), len(missing_secondary))

if missing_primary:
    logger.warning("Missing Stage3 PRIMARY sensors (showing up to 20): %s", missing_primary[:20])
if missing_secondary:
    logger.warning("Missing Stage3 SECONDARY sensors (showing up to 20): %s", missing_secondary[:20])


2026-06-16 20:44:26,796 | INFO | capstone.gold.cascade.stage3_improved | Stage3 missing sensors: primary=0 secondary=0


----

## Define the Primary Profile Breach Logic

This helper function counts how many Stage 3 primary rule sensors fall outside their reference profile bounds for each row.

The main idea is simple: if enough important primary sensors move outside the stored normal range, that row has stronger evidence that something abnormal is happening.

In [38]:

def compute_primary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name")[["lower_bound", "upper_bound"]]

    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns or feature_name not in reference_lookup.index:
            continue

        lower = reference_lookup.loc[feature_name, "lower_bound"]
        upper = reference_lookup.loc[feature_name, "upper_bound"]

        breach_flag = ((dataframe[feature_name] < lower) | (dataframe[feature_name] > upper)).astype(int)
        breach_counts = breach_counts + breach_flag

    breach_counts.name = "stage3_profile_breach_count"
    return breach_counts





## Define the Secondary Corroboration Logic

This helper function does a similar job for the secondary rule sensors.

The difference is that these secondary sensors are used more as corroboration than as the main profile breach signal. In other words, they help support the alert rather than carrying the same priority as the primary sensor group.

In [39]:
cascade_results["stage3_profile_breach_count"] = compute_primary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

----

## Compute the Secondary Breach Count

Here I apply the secondary breach logic to the cascade results dataframe.

This creates the row-level count of secondary rule sensors that move outside their reference profile bounds. Later this gets converted into a corroboration flag that contributes to the Stage 3 evidence count.

In [40]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

#### Compute Stage 3 secondary breach evidence

This cell performs the next transformation in `Compute the Secondary Breach Count`. Review the output or logged message before relying on this result downstream.

In [41]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

----

## Define the Persistence Logic

Not every abnormal point matters equally. Some are isolated spikes, while others persist across nearby rows.

This helper function checks for persistence by looking at recent Stage 2 flags inside a rolling window. If enough flags appear within that window, the row receives a persistence flag.

In [42]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

#### Compute Stage 3 persistence evidence

This cell performs the next transformation in `Define the Persistence Logic`. Review the output or logged message before relying on this result downstream.

In [43]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=STAGE3_ROLLING_WINDOW_SIZE,
    minimum_flags_in_window=STAGE3_MINIMUM_FLAGS_IN_WINDOW,
)


----

## Define the Drift Logic

This helper function checks for rolling drift behavior in the Stage 3 watch features.

The idea is to compare each feature to its recent rolling median and flag rows where the current value drifts far enough away from that recent local behavior. This gives Stage 3 another kind of evidence that is different from simple threshold breaches.

In [44]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

----

## Compute the Drift Flag Inputs

Here I build the combined Stage 3 watch feature list that will be used for the drift check.

This keeps the drift logic focused on the same important sensor groups already being watched by the Stage 3 rule layer.

In [45]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=STAGE3_DRIFT_ROLLING_WINDOW_SIZE,
    drift_threshold_multiplier=STAGE3_DRIFT_THRESHOLD_MULTIPLIER,
)

----

## Build the Final Stage 3 Evidence Flags and Final Cascade Decision

Now I combine the Stage 3 evidence into the final confirmation logic.

This section creates:
- the primary profile breach flag
- the secondary corroboration flag
- the persistence flag
- the drift flag
- the overall Stage 3 evidence count
- the final `cascade_final_flag`

The final cascade decision is intentionally strict. A row must first pass through Stage 1 and Stage 2, and then it must also show enough Stage 3 evidence to be considered a final cascade alert.

In [46]:
# Tunning Configs

# Relaxed:
RELAXED_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON = int(2)

# Medium
MEDIUM_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON = int(3)

# Strict
STRICT_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON = int(5)


#### Compute Stage 3 persistence evidence

This cell evaluates model behavior against the configured labels or thresholds. The resulting counts and metrics should match the saved JSON/CSV summaries used in Task 3.

In [47]:
# =========================================================
# Stage 3 tuning helpers
# =========================================================

def build_stage3_candidate_output(
    dataframe: pd.DataFrame,
    *,
    candidate_params: dict,
    stage3_watch_features: list[str],
) -> dict:
    candidate_dataframe = dataframe

    min_weighted_score = float(candidate_params["min_weighted_score"])
    rolling_window_size = int(candidate_params["rolling_window_size"])
    minimum_flags_in_window = int(candidate_params["minimum_flags_in_window"])
    strong_primary_hits = int(candidate_params["strong_primary_hits"])
    drift_threshold_multiplier = float(candidate_params["drift_threshold_multiplier"])

    profile_breach_flag = (
        candidate_dataframe["stage3_profile_breach_count"]
        >= STAGE3_MIN_PRIMARY_SENSOR_HITS
    ).astype(int)

    strong_primary_flag = (
        candidate_dataframe["stage3_profile_breach_count"]
        >= strong_primary_hits
    ).astype(int)

    corroboration_flag = (
        candidate_dataframe["stage3_secondary_breach_count"]
        >= STAGE3_MIN_SECONDARY_SENSOR_HITS
    ).astype(int)

    persistence_flag = compute_persistence_flag(
        candidate_dataframe["stage2_flag"],
        rolling_window_size=rolling_window_size,
        minimum_flags_in_window=minimum_flags_in_window,
    )

    drift_flag = compute_drift_flag(
        candidate_dataframe,
        feature_columns=stage3_watch_features,
        rolling_window_size=STAGE3_DRIFT_ROLLING_WINDOW_SIZE,
        drift_threshold_multiplier=drift_threshold_multiplier,
    )

    rule_evidence_count = (
        profile_breach_flag
        + persistence_flag
        + drift_flag
        + corroboration_flag
    )

    weighted_evidence_score = (
        profile_breach_flag * STAGE3_PROFILE_BREACH_WEIGHT
        + corroboration_flag * STAGE3_CORROBORATION_WEIGHT
        + persistence_flag * STAGE3_PERSISTENCE_WEIGHT
        + drift_flag * STAGE3_DRIFT_WEIGHT
    )

    confirmed_flag = (
        (strong_primary_flag == 1)
        | (
            (profile_breach_flag == 1)
            & (weighted_evidence_score >= min_weighted_score)
        )
    ).astype(int)

    final_flag = (
        (candidate_dataframe["stage2_flag"] == 1)
        & (confirmed_flag == 1)
    ).astype(int)

    return {
        "profile_breach_flag": profile_breach_flag,
        "strong_primary_flag": strong_primary_flag,
        "corroboration_flag": corroboration_flag,
        "persistence_flag": persistence_flag,
        "drift_flag": drift_flag,
        "rule_evidence_count": rule_evidence_count,
        "weighted_evidence_score": weighted_evidence_score,
        "confirmed_flag": confirmed_flag,
        "final_flag": final_flag,
    }


def evaluate_stage3_candidate(
    *,
    candidate_params: dict,
    candidate_output: dict,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    min_recall: float,
) -> dict:
    final_flags: np.ndarray = as_int_array(
        candidate_output["final_flag"],
        "final_flag",
    )

    test_mask_array_local: np.ndarray = test_mask.to_numpy(dtype=bool)

    final_alert_count_all_rows = int(final_flags.sum())
    final_alert_count_test_rows = int(final_flags[test_mask_array_local].sum())

    if test_labels is not None:
        test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)
        test_flags: np.ndarray = final_flags[test_mask_array_local].astype(int)

        precision, recall, f1, _ = precision_recall_fscore_support(
            test_labels_array,
            test_flags,
            average="binary",
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            test_labels_array,
            test_flags,
            labels=[0, 1],
        ).ravel()

        alert_rate = float(final_alert_count_test_rows) / float(max(len(test_flags), 1))

        if float(recall) < float(min_recall):
            selection_score = -1000.0 + float(recall)
        else:
            selection_score = (
                (3.0 * float(f1))
                + (1.0 * float(precision))
                - (1.0 * float(alert_rate))
            )
    else:
        precision = None
        recall = None
        f1 = None
        tn = None
        fp = None
        fn = None
        tp = None
        alert_rate = float(final_alert_count_test_rows) / float(
            max(int(test_mask_array_local.sum()), 1)
        )
        selection_score = -alert_rate

    return {
        **candidate_params,
        "final_alert_count_all_rows": final_alert_count_all_rows,
        "final_alert_count_test_rows": final_alert_count_test_rows,
        "alert_rate_test_rows": float(alert_rate),
        "precision": None if precision is None else float(precision),
        "recall": None if recall is None else float(recall),
        "f1": None if f1 is None else float(f1),
        "selection_score": float(selection_score),
        "tn": None if tn is None else int(tn),
        "fp": None if fp is None else int(fp),
        "fn": None if fn is None else int(fn),
        "tp": None if tp is None else int(tp),
    }


# =========================================================
# Build Stage 3 tuning candidate grid
# =========================================================

stage3_tuning_grid_normalized = {
    "min_weighted_score": [
        float(value)
        for value in STAGE3_TUNING_GRID.get("min_weighted_score", [STAGE3_MIN_WEIGHTED_SCORE])
    ],
    "rolling_window_size": [
        int(value)
        for value in STAGE3_TUNING_GRID.get("rolling_window_size", [STAGE3_ROLLING_WINDOW_SIZE])
    ],
    "minimum_flags_in_window": [
        int(value)
        for value in STAGE3_TUNING_GRID.get("minimum_flags_in_window", [STAGE3_MINIMUM_FLAGS_IN_WINDOW])
    ],
    "strong_primary_hits": [
        int(value)
        for value in STAGE3_TUNING_GRID.get("strong_primary_hits", [STAGE3_STRONG_PRIMARY_HITS])
    ],
    "drift_threshold_multiplier": [
        float(value)
        for value in STAGE3_TUNING_GRID.get("drift_threshold_multiplier", [STAGE3_DRIFT_THRESHOLD_MULTIPLIER])
    ],
}

stage3_candidate_rows = []
stage3_best_result = None
stage3_best_output = None

for (
    min_weighted_score,
    rolling_window_size,
    minimum_flags_in_window,
    strong_primary_hits,
    drift_threshold_multiplier,
) in product(
    stage3_tuning_grid_normalized["min_weighted_score"],
    stage3_tuning_grid_normalized["rolling_window_size"],
    stage3_tuning_grid_normalized["minimum_flags_in_window"],
    stage3_tuning_grid_normalized["strong_primary_hits"],
    stage3_tuning_grid_normalized["drift_threshold_multiplier"],
):
    candidate_params = {
        "min_weighted_score": min_weighted_score,
        "rolling_window_size": rolling_window_size,
        "minimum_flags_in_window": minimum_flags_in_window,
        "strong_primary_hits": strong_primary_hits,
        "drift_threshold_multiplier": drift_threshold_multiplier,
    }

    candidate_output = build_stage3_candidate_output(
        cascade_results,
        candidate_params=candidate_params,
        stage3_watch_features=stage3_rule_watch_features,
    )

    candidate_result = evaluate_stage3_candidate(
        candidate_params=candidate_params,
        candidate_output=candidate_output,
        test_mask=test_mask,
        test_labels=test_labels,
        min_recall=STAGE3_MIN_SELECTION_RECALL,
    )

    stage3_candidate_rows.append(candidate_result)

    if (
        stage3_best_result is None
        or candidate_result["selection_score"] > stage3_best_result["selection_score"]
    ):
        stage3_best_result = candidate_result
        stage3_best_output = candidate_output


if stage3_best_result is None or stage3_best_output is None:
    raise ValueError("Stage 3 tuning did not produce a valid best result.")

stage3_search_results = (
    pd.DataFrame(stage3_candidate_rows)
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

# =========================================================
# Apply selected Stage 3 output to cascade_results
# =========================================================

cascade_results["stage3_profile_breach_flag"] = stage3_best_output["profile_breach_flag"].astype(int)
cascade_results["stage3_strong_primary_flag"] = stage3_best_output["strong_primary_flag"].astype(int)
cascade_results["stage3_corroboration_flag"] = stage3_best_output["corroboration_flag"].astype(int)
cascade_results["stage3_persistence_flag"] = stage3_best_output["persistence_flag"].astype(int)
cascade_results["stage3_drift_flag"] = stage3_best_output["drift_flag"].astype(int)
cascade_results["stage3_rule_evidence_count"] = stage3_best_output["rule_evidence_count"].astype(int)
cascade_results["stage3_weighted_evidence_score"] = stage3_best_output["weighted_evidence_score"].astype(float)
cascade_results["stage3_weighted_score"] = cascade_results["stage3_weighted_evidence_score"]
cascade_results["stage3_confirmed_flag"] = stage3_best_output["confirmed_flag"].astype(int)

cascade_results["cascade_stage3_improved_flag"] = stage3_best_output["final_flag"].astype(int)

# 03C final model output should use the selected Stage 3 improved flag.
cascade_results["cascade_final_flag"] = cascade_results["cascade_stage3_improved_flag"].astype(int)

# Optional Stage 3 operating-mode variants for comparison.
cascade_results["cascade_stage3_relaxed_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (cascade_results["stage3_weighted_evidence_score"] >= 2.0)
).astype(int)

cascade_results["cascade_stage3_medium_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (cascade_results["stage3_weighted_evidence_score"] >= 3.0)
).astype(int)

cascade_results["cascade_stage3_strict_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (cascade_results["stage3_weighted_evidence_score"] >= 5.0)
).astype(int)

cascade_results["alert_priority"] = np.select(
    [
        (cascade_results["stage2_flag"] == 1)
        & (cascade_results["stage3_strong_primary_flag"] == 1),
        (cascade_results["stage2_flag"] == 1)
        & (cascade_results["stage3_confirmed_flag"] == 1),
        (cascade_results["stage2_flag"] == 1),
    ],
    [
        "high",
        "medium",
        "low",
    ],
    default="none",
)

cascade_results["stage3_priority_class"] = cascade_results["alert_priority"]

stage3_selected_params = {
    "min_weighted_score": float(stage3_best_result["min_weighted_score"]),
    "rolling_window_size": int(stage3_best_result["rolling_window_size"]),
    "minimum_flags_in_window": int(stage3_best_result["minimum_flags_in_window"]),
    "strong_primary_hits": int(stage3_best_result["strong_primary_hits"]),
    "drift_threshold_multiplier": float(stage3_best_result["drift_threshold_multiplier"]),
}

stage3_summary = {
    "stage3_variant": "tuned_confirmation_layer",
    "stage3_selected_params": stage3_selected_params,
    "stage3_search_candidate_count": int(len(stage3_search_results)),
    "stage3_selection_score": float(stage3_best_result["selection_score"]),
    "stage3_precision": stage3_best_result["precision"],
    "stage3_recall": stage3_best_result["recall"],
    "stage3_f1": stage3_best_result["f1"],
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "strong_primary_rows_all": int(cascade_results["stage3_strong_primary_flag"].sum()),
    "strong_primary_rows_test": int(cascade_results.loc[test_mask, "stage3_strong_primary_flag"].sum()),
    "profile_breach_rows_all": int(cascade_results["stage3_profile_breach_flag"].sum()),
    "profile_breach_rows_test": int(cascade_results.loc[test_mask, "stage3_profile_breach_flag"].sum()),
    "corroboration_rows_all": int(cascade_results["stage3_corroboration_flag"].sum()),
    "corroboration_rows_test": int(cascade_results.loc[test_mask, "stage3_corroboration_flag"].sum()),
    "persistence_rows_all": int(cascade_results["stage3_persistence_flag"].sum()),
    "persistence_rows_test": int(cascade_results.loc[test_mask, "stage3_persistence_flag"].sum()),
    "drift_rows_all": int(cascade_results["stage3_drift_flag"].sum()),
    "drift_rows_test": int(cascade_results.loc[test_mask, "stage3_drift_flag"].sum()),
    "weighted_score_mean_all": float(cascade_results["stage3_weighted_evidence_score"].mean()),
    "weighted_score_mean_test": float(cascade_results.loc[test_mask, "stage3_weighted_evidence_score"].mean()),
    "confirmed_rows_all": int(cascade_results["stage3_confirmed_flag"].sum()),
    "confirmed_rows_test": int(cascade_results.loc[test_mask, "stage3_confirmed_flag"].sum()),
    "final_stage3_improved_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_stage3_improved_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
    "stage3_relaxed_alert_count_all_rows": int(cascade_results["cascade_stage3_relaxed_flag"].sum()),
    "stage3_relaxed_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum()),
    "stage3_medium_alert_count_all_rows": int(cascade_results["cascade_stage3_medium_flag"].sum()),
    "stage3_medium_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum()),
    "stage3_strict_alert_count_all_rows": int(cascade_results["cascade_stage3_strict_flag"].sum()),
    "stage3_strict_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum()),
    "high_priority_rows_all": int((cascade_results["alert_priority"] == "high").sum()),
    "high_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "high").sum()),
    "medium_priority_rows_all": int((cascade_results["alert_priority"] == "medium").sum()),
    "medium_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "medium").sum()),
    "low_priority_rows_all": int((cascade_results["alert_priority"] == "low").sum()),
    "low_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "low").sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_tuning",
    message="Applied Stage 3 tuning grid, selected the best confirmation configuration, and promoted the selected Stage 3 flag to the final cascade output.",
    data=stage3_summary,
    logger=logger,
)

display(stage3_search_results.head(15))
display(cascade_results.head(5))

2026-06-16 20:46:22,189 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:46:22.189340+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage3_tuning', 'message': 'Applied Stage 3 tuning grid, selected the best confirmation configuration, and promoted the selected Stage 3 flag to the final cascade output.', 'why': None, 'consequence': None, 'data': {'stage3_variant': 'tuned_confirmation_layer', 'stage3_selected_params': {'min_weighted_score': 3.25, 'rolling_window_size': 3, 'minimum_flags_in_window': 3, 'strong_primary_hits': 3, 'drift_threshold_multiplier': 1.5}, 'stage3_search_candidate_count': 120, 'stage3_selection_score': -0.005413664701617665, 'stage3_precision': 0.010615711252653927, 'stage3_recall': 0.5932203389830508, 'stage3_f1': 0.020858164481525627, 'primary_rule_sensor_count': 5, 'secondary_rule_sensor_count': 8, 'strong_primary_rows_all': 48841, 'strong_primary_rows_test': 2145

,min_weighted_score,rolling_window_size,minimum_flags_in_window,strong_primary_hits,drift_threshold_multiplier,final_alert_count_all_rows,final_alert_count_test_rows,alert_rate_test_rows,precision,recall,f1,selection_score,tn,fp,fn,tp
0,3.25,3,3,3,1.50,37024,6594,0.078604,0.010616,0.593220,0.020858,-0.005414,77247,6524,48,70
1,3.50,3,3,3,1.50,37024,6594,0.078604,0.010616,0.593220,0.020858,-0.005414,77247,6524,48,70
2,4.00,3,3,3,1.50,37024,6594,0.078604,0.010616,0.593220,0.020858,-0.005414,77247,6524,48,70
3,3.75,3,3,3,1.50,37024,6594,0.078604,0.010616,0.593220,0.020858,-0.005414,77247,6524,48,70
4,3.75,3,3,3,1.25,37025,6595,0.078616,0.010614,0.593220,0.020855,-0.005437,77246,6525,48,70
5,4.00,3,3,3,1.25,37025,6595,0.078616,0.010614,0.593220,0.020855,-0.005437,77246,6525,48,70
6,3.50,3,3,3,1.25,37025,6595,0.078616,0.010614,0.593220,0.020855,-0.005437,77246,6525,48,70
7,3.25,3,3,3,1.25,37025,6595,0.078616,0.010614,0.593220,0.020855,-0.005437,77246,6525,48,70
8,3.25,3,3,3,1.00,37026,6596,0.078628,0.010612,0.593220,0.020852,-0.005459,77245,6526,48,70
9,3.50,3,3,3,1.00,37026,6596,0.078628,0.010612,0.593220,0.020852,-0.005459,77245,6526,48,70


,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,plot_order_index,stage1_threshold,stage1_threshold_percentile,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_strong_primary_flag,stage3_corroboration_flag,stage3_rule_evidence_count,stage3_weighted_evidence_score,stage3_weighted_score,stage3_confirmed_flag,cascade_stage3_improved_flag,cascade_final_flag,cascade_stage3_relaxed_flag,cascade_stage3_medium_flag,cascade_stage3_strict_flag,alert_priority,stage3_priority_class
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,14598431322315673869,run__001,sensor.csv,0,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True,0.409368,0.090632,1,0,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,0,1,1,1.0,1.0,0,0,0,0,0,0,none,none
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,15954729095895098000,run__001,sensor.csv,1,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True,0.409368,0.090632,1,0,1,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,0,1,1,1.0,1.0,0,0,0,0,0,0,none,none
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,10041703297090838359,run__001,sensor.csv,2,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.280002,-0.565216,0.464286,1.307690,0.578311,-0.537436,-0.630443,-0.078128,0.859651,-0.666728,-1.160344,1.006862,0.146473,-0.108279,0.230324,-0.259305,-0.439206,-0.506023,0.044628,0.300410,0.019647,-0.099876,0.178490,0.225476,-0.356118,-0.460964,-0.049008,-1.791147,-0.158283,...,False,0.354679,0.470583,Fal

### Ask

How should I think about the final Stage 3 rule?

### Answer

Stage 3 is the confirmation layer that tries to reduce weak or isolated model alerts.

In this notebook, a row can become a final cascade alert in one of two main ways:
- it shows enough primary profile breaches on its own
- or it builds enough combined rule evidence across the Stage 3 checks

So Stage 3 is not replacing the model stages. It is acting as a final confirmation layer after the row has already passed through both model filters.

----

In [48]:

cascade_results = finalize_stage_flag_columns(
    cascade_results,
    stage_names=["stage1", "stage2", "stage3"],
)

if "stage3_confirmed_flag" in cascade_results.columns and "stage3_flag" not in cascade_results.columns:
    cascade_results["stage3_flag"] = cascade_results["stage3_confirmed_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="finalize_stage_flags",
    message="Finalized sparse cascade stage flags after Stage 3 rule processing.",
    data={
        "stage1_flag_count": int(cascade_results["stage1_flag"].sum()) if "stage1_flag" in cascade_results.columns else 0,
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()) if "stage2_flag" in cascade_results.columns else 0,
        "stage3_flag_count": int(cascade_results["stage3_flag"].sum()) if "stage3_flag" in cascade_results.columns else 0,
        "cascade_final_flag_count": int(cascade_results["cascade_final_flag"].sum()) if "cascade_final_flag" in cascade_results.columns else 0,
    },
    logger=logger,
)

2026-06-16 20:46:24,598 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:46:24.598119+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_stage_flags', 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.', 'why': None, 'consequence': None, 'data': {'stage1_flag_count': 103054, 'stage2_flag_count': 55684, 'stage3_flag_count': 58763, 'cascade_final_flag_count': 37024}}


{'ts_utc': '2026-06-16T20:46:24.598119+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_stage_flags',
 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.',
 'why': None,
 'consequence': None,
 'data': {'stage1_flag_count': 103054,
  'stage2_flag_count': 55684,
  'stage3_flag_count': 58763,
  'cascade_final_flag_count': 37024}}

## Build the Main Cascade Metrics

Here I summarize the main tuned cascade results.

This includes:
- Stage 1 alert counts
- Stage 2 alert counts
- final cascade alert counts
- test-row alert counts
- evaluation metrics such as precision, recall, and F1 when test labels are available

This is the main performance snapshot for the tuned cascade notebook.

In [49]:
cascade_metrics: dict[str, Any] = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
}

if test_labels is not None:
    test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)

    cascade_test_flags: np.ndarray = (
        cascade_results
        .loc[test_mask, "cascade_final_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels_array,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

display(cascade_metrics)

{'model': '3-Stage Cascade',
 'stage1_alert_count_all_rows': 103054,
 'stage2_alert_count_all_rows': 55684,
 'final_alert_count_all_rows': 37024,
 'stage1_alert_count_test_rows': 43236,
 'stage2_alert_count_test_rows': 16108,
 'final_alert_count_test_rows': 6594,
 'precision': 0.010615711252653927,
 'recall': 0.5932203389830508,
 'f1': 0.020858164481525627}

### Ask

What should I pay attention to in the tuned cascade metrics?

### Answer

The main things I care about are:
- how many rows survive from Stage 1 into Stage 2
- how many rows survive from Stage 2 into the final cascade output
- whether the tuned Stage 2 branch changed the final alert set in a useful way
- precision, recall, and F1 on the test rows when labels are available

For this notebook, the key question is not just whether the cascade detects anomalies. It is whether the tuned Stage 2 search helps produce a cleaner final alert set than the default cascade and the baseline.

---

In [50]:
# =========================================================
# Cascade output validation on held-out test rows
# =========================================================

def validate_cascade_output(
    results_dataframe: pd.DataFrame,
    *,
    test_mask: pd.Series,
    label_column: str = "anomaly_flag",
    row_id_column: str = "meta__row_id",
    final_flag_column: str = "cascade_final_flag",
) -> dict[str, Any]:
    required_columns = [
        row_id_column,
        "meta__is_train_flag",
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        final_flag_column,
    ]

    for column_name in required_columns:
        if column_name not in results_dataframe.columns:
            raise ValueError(f"Missing required cascade output column: {column_name}")

    if len(test_mask) != len(results_dataframe):
        raise ValueError("test_mask length does not match cascade results dataframe.")

    if not results_dataframe[row_id_column].is_unique:
        raise ValueError(f"{row_id_column} is not unique in cascade results.")

    binary_columns = [
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        final_flag_column,
    ]

    for column_name in binary_columns:
        invalid_values = sorted(
            set(results_dataframe[column_name].dropna().unique()) - {0, 1}
        )

        if invalid_values:
            raise ValueError(f"{column_name} contains non-binary values: {invalid_values}")

    bad_stage2_gate_rows = results_dataframe.loc[
        (results_dataframe["stage2_flag"] == 1)
        & (results_dataframe["stage1_flag"] != 1)
    ]

    if len(bad_stage2_gate_rows) > 0:
        raise ValueError(
            "Stage 2 gate violation: stage2_flag has rows that did not pass stage1_flag. "
            f"Bad row count: {len(bad_stage2_gate_rows)}"
        )

    bad_final_gate_rows = results_dataframe.loc[
        (results_dataframe[final_flag_column] == 1)
        & (results_dataframe["stage2_flag"] != 1)
    ]

    if len(bad_final_gate_rows) > 0:
        raise ValueError(
            f"Final cascade gate violation: {final_flag_column} has rows that did not pass stage2_flag. "
            f"Bad row count: {len(bad_final_gate_rows)}"
        )

    test_dataframe = results_dataframe.loc[test_mask].copy()

    validation_summary: dict[str, Any] = {
        "all_rows": int(len(results_dataframe)),
        "test_rows": int(len(test_dataframe)),
        "stage1_alert_count_all_rows": int(results_dataframe["stage1_flag"].sum()),
        "stage1_alert_count_test_rows": int(test_dataframe["stage1_flag"].sum()),
        "stage2_raw_alert_count_all_rows": int(results_dataframe["stage2_raw_flag"].sum()),
        "stage2_raw_alert_count_test_rows": int(test_dataframe["stage2_raw_flag"].sum()),
        "stage2_alert_count_all_rows": int(results_dataframe["stage2_flag"].sum()),
        "stage2_alert_count_test_rows": int(test_dataframe["stage2_flag"].sum()),
        "final_alert_count_all_rows": int(results_dataframe[final_flag_column].sum()),
        "final_alert_count_test_rows": int(test_dataframe[final_flag_column].sum()),
    }

    if label_column in results_dataframe.columns:
        y_true: np.ndarray = (
            test_dataframe[label_column]
            .fillna(0)
            .astype(int)
            .to_numpy(dtype=int)
        )

        y_pred: np.ndarray = (
            test_dataframe[final_flag_column]
            .fillna(0)
            .astype(int)
            .to_numpy(dtype=int)
        )

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        validation_summary.update(
            {
                "test_anomaly_rows": int(y_true.sum()),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )

    print("Cascade output validation passed.")
    return validation_summary

#### Validate cascade output columns and counts

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [51]:

cascade_output_validation = validate_cascade_output(
    cascade_results,
    test_mask=test_mask,
    final_flag_column="cascade_final_flag",
)

ledger.add(
    kind="step",
    step="validate_cascade_output",
    message="Validated cascade staged outputs and final held-out test metrics.",
    data=cascade_output_validation,
    logger=logger,
)

display(pd.DataFrame([cascade_output_validation]))

2026-06-16 20:46:26,145 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:46:26.145038+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_cascade_output', 'message': 'Validated cascade staged outputs and final held-out test metrics.', 'why': None, 'consequence': None, 'data': {'all_rows': 220320, 'test_rows': 83889, 'stage1_alert_count_all_rows': 103054, 'stage1_alert_count_test_rows': 43236, 'stage2_raw_alert_count_all_rows': 55684, 'stage2_raw_alert_count_test_rows': 16108, 'stage2_alert_count_all_rows': 55684, 'stage2_alert_count_test_rows': 16108, 'final_alert_count_all_rows': 37024, 'final_alert_count_test_rows': 6594, 'test_anomaly_rows': 118, 'precision': 0.010615711252653927, 'recall': 0.5932203389830508, 'f1': 0.020858164481525627, 'tn': 77247, 'fp': 6524, 'fn': 48, 'tp': 70}}


Cascade output validation passed.


,all_rows,test_rows,stage1_alert_count_all_rows,stage1_alert_count_test_rows,stage2_raw_alert_count_all_rows,stage2_raw_alert_count_test_rows,stage2_alert_count_all_rows,stage2_alert_count_test_rows,final_alert_count_all_rows,final_alert_count_test_rows,test_anomaly_rows,precision,recall,f1,tn,fp,fn,tp
0,220320,83889,103054,43236,55684,16108,55684,16108,37024,6594,118,0.010616,0.59322,0.020858,77247,6524,48,70


#### Run validation guardrails

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [52]:
# =========================================================
# Validate 03c Stage 3 variant gates
# =========================================================

stage3_variant_columns = [
    "cascade_stage3_relaxed_flag",
    "cascade_stage3_medium_flag",
    "cascade_stage3_strict_flag",
]

for variant_column in stage3_variant_columns:
    if variant_column not in cascade_results.columns:
        raise ValueError(f"Missing Stage 3 variant column: {variant_column}")

    invalid_values = sorted(set(cascade_results[variant_column].dropna().unique()) - {0, 1})

    if invalid_values:
        raise ValueError(f"{variant_column} contains non-binary values: {invalid_values}")

    bad_variant_gate_rows = cascade_results.loc[
        (cascade_results[variant_column] == 1)
        & (cascade_results["stage2_flag"] != 1)
    ]

    if len(bad_variant_gate_rows) > 0:
        raise ValueError(
            f"{variant_column} gate violation: variant flag has rows that did not pass stage2_flag. "
            f"Bad row count: {len(bad_variant_gate_rows)}"
        )

stage3_variant_validation = {
    "stage3_relaxed_alert_count_all_rows": int(cascade_results["cascade_stage3_relaxed_flag"].sum()),
    "stage3_medium_alert_count_all_rows": int(cascade_results["cascade_stage3_medium_flag"].sum()),
    "stage3_strict_alert_count_all_rows": int(cascade_results["cascade_stage3_strict_flag"].sum()),
    "stage3_relaxed_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum()),
    "stage3_medium_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum()),
    "stage3_strict_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum()),
}

ledger.add(
    kind="step",
    step="validate_stage3_variant_outputs",
    message="Validated Stage 3 relaxed, medium, and strict variant gates.",
    data=stage3_variant_validation,
    logger=logger,
)

display(pd.DataFrame([stage3_variant_validation]))

2026-06-16 20:46:26,535 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:46:26.535108+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_stage3_variant_outputs', 'message': 'Validated Stage 3 relaxed, medium, and strict variant gates.', 'why': None, 'consequence': None, 'data': {'stage3_relaxed_alert_count_all_rows': 52913, 'stage3_medium_alert_count_all_rows': 38213, 'stage3_strict_alert_count_all_rows': 210, 'stage3_relaxed_alert_count_test_rows': 13713, 'stage3_medium_alert_count_test_rows': 7286, 'stage3_strict_alert_count_test_rows': 61}}


,stage3_relaxed_alert_count_all_rows,stage3_medium_alert_count_all_rows,stage3_strict_alert_count_all_rows,stage3_relaxed_alert_count_test_rows,stage3_medium_alert_count_test_rows,stage3_strict_alert_count_test_rows
0,52913,38213,210,13713,7286,61


----

In [53]:
# =========================================================
# Evaluate 03C Stage 3 operating-mode metrics
# =========================================================
# The previous validation cell confirms that relaxed, medium, and strict flags
# exist and are gated correctly. This cell evaluates each operating mode against
# the held-out test labels so Gold 04 can compare full precision/recall/F1
# instead of only alert counts.

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)


def _resolve_stage3_variant_flag_column(
    dataframe: pd.DataFrame,
    mode_name: str,
) -> str:
    """
    Resolve the flag column for a Stage 3 operating mode.

    Parameters
    ----------
    dataframe:
        Cascade results dataframe.
    mode_name:
        Operating mode name: relaxed, medium, or strict.

    Returns
    -------
    str
        Column name containing the binary alert flag for the requested mode.
    """
    candidate_columns = [
        f"cascade_stage3_{mode_name}_flag",
        f"stage3_{mode_name}_flag",
        f"final_stage3_{mode_name}_flag",
        f"{mode_name}_flag",
    ]

    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name

    available_flag_columns = [
        column_name
        for column_name in dataframe.columns
        if "stage3" in str(column_name).lower()
        and "flag" in str(column_name).lower()
    ]

    raise KeyError(
        f"Could not resolve Stage 3 {mode_name!r} flag column. "
        f"Checked: {candidate_columns}. "
        f"Available Stage 3 flag-like columns: {available_flag_columns}"
    )


def _resolve_stage3_score_column(
    dataframe: pd.DataFrame,
) -> str | None:
    """
    Resolve a continuous Stage 3 score column when available.

    ROC AUC and PR AUC are more meaningful when calculated from a continuous
    evidence score. If no score column is available, the evaluation function
    falls back to using the binary flag as the score.
    """
    candidate_columns = [
        "cascade_stage3_weighted_score",
        "stage3_weighted_score",
        "weighted_score",
        "final_stage3_score",
        "stage3_score",
        "cascade_stage3_score",
    ]

    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name

    return None


def _resolve_label_column(
    dataframe: pd.DataFrame,
) -> str:
    """
    Resolve the anomaly label column used for model evaluation.

    The notebook may define TARGET_FLAG_COLUMN in some runs, but Pylance cannot
    safely infer that from globals(). This function reads it through globals()
    directly to avoid undefined-variable diagnostics.
    """
    target_flag_column_value = globals().get("TARGET_FLAG_COLUMN")

    candidate_columns = [
        "anomaly_flag",
        "target_flag",
        "is_anomaly",
        "label",
        "machine_failure_flag",
    ]

    if target_flag_column_value is not None:
        candidate_columns.insert(0, str(target_flag_column_value))

    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name

    raise KeyError(
        "Could not resolve anomaly label column. "
        f"Checked: {candidate_columns}. "
        f"Available columns: {list(dataframe.columns)}"
    )


def _safe_binary_auc(
    y_true: np.ndarray,
    y_score: np.ndarray,
    metric_name: str,
) -> float | None:
    """
    Compute ROC AUC or PR AUC when both classes are present.
    """
    unique_labels = np.unique(y_true)

    if len(unique_labels) < 2:
        return None

    if metric_name == "roc_auc":
        return float(roc_auc_score(y_true, y_score))

    if metric_name == "pr_auc":
        return float(average_precision_score(y_true, y_score))

    raise ValueError(f"Unsupported metric_name: {metric_name}")


def evaluate_stage3_operating_mode(
    *,
    results_dataframe: pd.DataFrame,
    test_mask: pd.Series,
    mode_name: str,
) -> dict[str, Any]:
    """
    Evaluate one Stage 3 operating mode against held-out test labels.

    Returns precision, recall, F1, confusion-matrix counts, ROC AUC, PR AUC,
    and alert counts for the selected mode.
    """
    flag_column = _resolve_stage3_variant_flag_column(
        dataframe=results_dataframe,
        mode_name=mode_name,
    )

    label_column = _resolve_label_column(results_dataframe)

    if isinstance(test_mask, pd.Series):
        aligned_test_mask = test_mask.reindex(results_dataframe.index).fillna(False)
    else:
        aligned_test_mask = pd.Series(
            test_mask,
            index=results_dataframe.index,
        )

    test_dataframe = results_dataframe.loc[aligned_test_mask].copy()

    y_true = (
        test_dataframe[label_column]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

    y_pred = (
        test_dataframe[flag_column]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    score_column = _resolve_stage3_score_column(results_dataframe)

    if score_column is not None:
        y_score = (
            test_dataframe[score_column]
            .fillna(0)
            .astype(float)
            .to_numpy(dtype=float)
        )
    else:
        y_score = y_pred.astype(float)

    roc_auc = _safe_binary_auc(y_true, y_score, "roc_auc")
    pr_auc = _safe_binary_auc(y_true, y_score, "pr_auc")

    return {
        "mode": mode_name,
        "flag_column": flag_column,
        "score_column": score_column,
        "alert_count_all_rows": int(results_dataframe[flag_column].fillna(0).astype(int).sum()),
        "alert_count_test_rows": int(y_pred.sum()),
        "test_row_count": int(len(test_dataframe)),
        "test_anomaly_rows": int(y_true.sum()),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


stage3_operating_mode_metrics = {
    mode_name: evaluate_stage3_operating_mode(
        results_dataframe=cascade_results,
        test_mask=test_mask,
        mode_name=mode_name,
    )
    for mode_name in ["relaxed", "medium", "strict"]
}

stage3_variant_metric_summary = {}

for mode_name, mode_metrics in stage3_operating_mode_metrics.items():
    prefix = f"stage3_{mode_name}"

    stage3_variant_metric_summary.update(
        {
            f"{prefix}_precision": mode_metrics["precision"],
            f"{prefix}_recall": mode_metrics["recall"],
            f"{prefix}_f1": mode_metrics["f1"],
            f"{prefix}_roc_auc": mode_metrics["roc_auc"],
            f"{prefix}_pr_auc": mode_metrics["pr_auc"],
            f"{prefix}_tn": mode_metrics["tn"],
            f"{prefix}_fp": mode_metrics["fp"],
            f"{prefix}_fn": mode_metrics["fn"],
            f"{prefix}_tp": mode_metrics["tp"],
            f"{prefix}_flag_column": mode_metrics["flag_column"],
            f"{prefix}_score_column": mode_metrics["score_column"],
        }
    )

# Gold 04 reads from cascade_metrics, so update it before the summary JSON is saved.
cascade_metrics.update(stage3_variant_metric_summary)

ledger.add(
    kind="step",
    step="evaluate_stage3_operating_mode_metrics",
    message=(
        "Evaluated precision, recall, F1, ROC AUC, PR AUC, and confusion-matrix "
        "metrics for Stage 3 relaxed, medium, and strict operating modes."
    ),
    data=stage3_variant_metric_summary,
    logger=logger,
)

display(
    pd.DataFrame(stage3_operating_mode_metrics.values())
)

2026-06-16 20:46:28,012 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:46:28.012164+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'evaluate_stage3_operating_mode_metrics', 'message': 'Evaluated precision, recall, F1, ROC AUC, PR AUC, and confusion-matrix metrics for Stage 3 relaxed, medium, and strict operating modes.', 'why': None, 'consequence': None, 'data': {'stage3_relaxed_precision': 0.005833880259607672, 'stage3_relaxed_recall': 0.6779661016949152, 'stage3_relaxed_f1': 0.01156821632564529, 'stage3_relaxed_roc_auc': 0.8508693696637463, 'stage3_relaxed_pr_auc': 0.10725219643480478, 'stage3_relaxed_tn': 70138, 'stage3_relaxed_fp': 13633, 'stage3_relaxed_fn': 38, 'stage3_relaxed_tp': 80, 'stage3_relaxed_flag_column': 'cascade_stage3_relaxed_flag', 'stage3_relaxed_score_column': 'stage3_weighted_score', 'stage3_medium_precision': 0.010705462530881142, 'stage3_medium_recall': 0.6610169491525424, '

,mode,flag_column,score_column,alert_count_all_rows,alert_count_test_rows,test_row_count,test_anomaly_rows,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,relaxed,cascade_stage3_relaxed_flag,stage3_weighted_score,52913,13713,83889,118,0.005834,0.677966,0.011568,0.850869,0.107252,70138,13633,38,80
1,medium,cascade_stage3_medium_flag,stage3_weighted_score,38213,7286,83889,118,0.010705,0.661017,0.021070,0.850869,0.107252,76563,7208,40,78
2,strict,cascade_stage3_strict_flag,stage3_weighted_score,210,61,83889,118,0.442623,0.228814,0.301676,0.850869,0.107252,83737,34,91,27


#### Merge operating-mode metrics into summary payloads

This cell performs the next transformation in `----`. Review the output or logged message before relying on this result downstream.

In [54]:
# Push Stage 3 operating-mode metrics into the main metric containers
# before the summary JSON is built and saved.
cascade_metrics.update(stage3_variant_metric_summary)

stage3_summary.update(stage3_variant_metric_summary)
stage3_summary["stage3_operating_mode_metrics"] = stage3_operating_mode_metrics

----

## Build the Cascade Summary, Threshold Records, and Truth Artifact

Now I convert the tuned cascade results into formal pipeline artifacts.

This section does several important things:
- summarizes the cascade thresholds and metrics
- records the tuned Stage 2 selection details
- builds the main cascade summary record
- creates the Gold cascade truth record
- stores runtime facts and artifact paths
- links the cascade output back to the parent Gold truth
- prepares the final saved deliverables for downstream comparison

At this point, the notebook output becomes more than just a scored dataframe. It becomes a tracked modeling artifact in the pipeline.

----

In [55]:
# =========================================================
# Build 03C Stage 3 Improved summary, metadata, truth, and saved artifacts
# =========================================================

stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(
    cascade_results.loc[test_mask, "cascade_final_flag"].sum()
)

stage3_relaxed_alert_count_all_rows = int(
    cascade_results["cascade_stage3_relaxed_flag"].sum()
)
stage3_medium_alert_count_all_rows = int(
    cascade_results["cascade_stage3_medium_flag"].sum()
)
stage3_strict_alert_count_all_rows = int(
    cascade_results["cascade_stage3_strict_flag"].sum()
)

stage3_relaxed_alert_count_test_rows = int(
    cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum()
)
stage3_medium_alert_count_test_rows = int(
    cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum()
)
stage3_strict_alert_count_test_rows = int(
    cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum()
)

stage3_search_results_object = globals().get("stage3_search_results")
if isinstance(stage3_search_results_object, pd.DataFrame):
    stage3_search_candidate_count = int(len(stage3_search_results_object))
else:
    stage3_search_candidate_count = 0

stage3_selected_params_object = globals().get("stage3_selected_params")
if not isinstance(stage3_selected_params_object, dict):
    stage3_selected_params_object = {}

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

# =========================================================
# 03C artifact path payload
# =========================================================

cascade_stage3_improved_artifact_paths = {
    "cascade_stage3_improved_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
    "cascade_stage3_improved_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
    "cascade_stage3_improved_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
    "cascade_stage3_improved_stage1_models_path": str(STAGE1_MODELS_PATH),
    "cascade_stage3_improved_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
    "cascade_stage3_improved_stage2_models_path": str(STAGE2_MODELS_PATH),
    "cascade_stage3_improved_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_stage3_improved_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_stage3_improved_metadata_path": str(CASCADE_METADATA_PATH),
    "cascade_stage3_improved_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
}

# =========================================================
# Threshold record
# =========================================================

cascade_thresholds = {
    "cascade_variant": CASCADE_VARIANT,
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "stage2_threshold": float(stage2_threshold),
    "stage2_best_params": stage2_best_params,
    "stage3_variant": "tuned_confirmation_layer",
    "stage3_selected_params": stage3_selected_params_object,
    "stage3_search_candidate_count": stage3_search_candidate_count,
    "stage3_min_selection_recall": float(STAGE3_MIN_SELECTION_RECALL),
    "stage3_profile_breach_weight": float(STAGE3_PROFILE_BREACH_WEIGHT),
    "stage3_corroboration_weight": float(STAGE3_CORROBORATION_WEIGHT),
    "stage3_persistence_weight": float(STAGE3_PERSISTENCE_WEIGHT),
    "stage3_drift_weight": float(STAGE3_DRIFT_WEIGHT),
    "stage3_drift_rolling_window_size": int(STAGE3_DRIFT_ROLLING_WINDOW_SIZE),
}

# =========================================================
# Summary record
# =========================================================

cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_variant": CASCADE_VARIANT,
    "stage3_variant": "tuned_confirmation_layer",
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "stage3_relaxed_alert_count_all_rows": stage3_relaxed_alert_count_all_rows,
    "stage3_medium_alert_count_all_rows": stage3_medium_alert_count_all_rows,
    "stage3_strict_alert_count_all_rows": stage3_strict_alert_count_all_rows,
    "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
    "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
    "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
    **stage3_variant_metric_summary,
    "stage3_operating_mode_metrics": stage3_operating_mode_metrics,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "stage2_best_params": stage2_best_params,
    "stage2_search_candidate_count": int(len(stage2_search_results)),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage3_selected_params": stage3_selected_params_object,
    "stage3_search_candidate_count": stage3_search_candidate_count,
    "stage3_summary": stage3_summary,
}

stage3_summary.update(stage3_variant_metric_summary)
stage3_summary["stage3_operating_mode_metrics"] = stage3_operating_mode_metrics


# =========================================================
# Build cascade truth record
# =========================================================

truth_config_object = globals().get("TRUTH_CONFIG")
run_mode_value = globals().get("RUN_MODE")
config_profile_value = globals().get("CONFIG_PROFILE", "default")
gold_process_run_id_value = globals().get("GOLD_PROCESS_RUN_ID")

if isinstance(truth_config_object, dict):
    truth_config_snapshot = truth_config_object
else:
    truth_config_snapshot = {
        "runtime": {
            "stage": "gold_cascade",
            "dataset": DATASET_NAME,
            "cascade_variant": CASCADE_VARIANT,
            "mode": run_mode_value,
            "profile": config_profile_value,
        }
    }

cascade_truth_layer_name = "gold_cascade"

if isinstance(gold_process_run_id_value, str) and gold_process_run_id_value.strip():
    cascade_process_run_id = gold_process_run_id_value
else:
    cascade_process_run_id = make_process_run_id("gold_cascade_process")

cascade_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
    process_run_id=cascade_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "config_snapshot",
    truth_config_snapshot,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "runtime_facts",
    {
        "cascade_variant": CASCADE_VARIANT,
        "stage3_variant": "tuned_confirmation_layer",
        "result_row_count": int(len(cascade_results)),
        "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
        "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
        "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
        "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
        "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
        "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
        "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
        "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_best_params": stage2_best_params,
        "stage2_search_candidate_count": int(len(stage2_search_results)),
        "stage3_selected_params": stage3_selected_params_object,
        "stage3_search_candidate_count": stage3_search_candidate_count,
    },
)

cascade_truth = update_truth_section(
    cascade_truth,
    "artifact_paths",
    {
        "cascade_variant": CASCADE_VARIANT,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        **cascade_stage3_improved_artifact_paths,
    },
)

cascade_meta_columns = sorted(
    set(
        identify_meta_columns(cascade_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

cascade_feature_columns = identify_feature_columns(cascade_results)

cascade_truth_record = build_truth_record(
    truth_base=cascade_truth,
    row_count=len(cascade_results),
    column_count=cascade_results.shape[1] + 3,
    meta_columns=cascade_meta_columns,
    feature_columns=cascade_feature_columns,
)

CASCADE_TRUTH_HASH = cascade_truth_record["truth_hash"]

cascade_results = stamp_truth_columns(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

cascade_truth_path = save_truth_record(
    cascade_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
)

append_truth_index(
    cascade_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

# =========================================================
# Add lineage to summary and metadata
# =========================================================

cascade_summary["cascade_truth_hash"] = CASCADE_TRUTH_HASH
cascade_summary["cascade_truth_path"] = str(cascade_truth_path)
cascade_summary["cascade_process_run_id"] = cascade_process_run_id
cascade_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
cascade_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
cascade_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
cascade_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

display(cascade_summary)

cascade_metadata = {
    "cascade_variant": CASCADE_VARIANT,
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    **cascade_stage3_improved_artifact_paths,
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "stage2_best_params": stage2_best_params,
    "stage2_search_candidate_count": int(len(stage2_search_results)),
    "stage3_selected_params": stage3_selected_params_object,
    "stage3_search_candidate_count": stage3_search_candidate_count,
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "cascade_truth_path": str(cascade_truth_path),
    "cascade_process_run_id": cascade_process_run_id,
    "stage3_variant": "tuned_confirmation_layer",
}

display(cascade_metadata)

# =========================================================
# Save artifacts
# =========================================================

for output_path in [
    CASCADE_RESULTS_PATH_CSV,
    CASCADE_RESULTS_PATH_PICKLE,
    CASCADE_REFERENCE_PROFILE_PATH,
    STAGE1_MODEL_ARTIFACT_PATH,
    STAGE1_MODELS_PATH,
    STAGE2_MODEL_ARTIFACT_PATH,
    STAGE2_MODELS_PATH,
    CASCADE_THRESHOLDS_PATH,
    CASCADE_SUMMARY_PATH,
    CASCADE_METADATA_PATH,
]:
    output_path.parent.mkdir(parents=True, exist_ok=True)

cascade_results.to_csv(CASCADE_RESULTS_PATH_CSV, index=False)
cascade_results.to_pickle(CASCADE_RESULTS_PATH_PICKLE)

display(cascade_results)


display(reference_profile)
reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)

joblib.dump(stage1_model, STAGE1_MODEL_ARTIFACT_PATH)
joblib.dump(stage1_model, STAGE1_MODELS_PATH)

joblib.dump(stage2_model, STAGE2_MODEL_ARTIFACT_PATH)
joblib.dump(stage2_model, STAGE2_MODELS_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

for wandb_path in [
    CASCADE_RESULTS_PATH_CSV,
    CASCADE_RESULTS_PATH_PICKLE,
    CASCADE_REFERENCE_PROFILE_PATH,
    STAGE1_MODEL_ARTIFACT_PATH,
    STAGE1_MODELS_PATH,
    STAGE2_MODEL_ARTIFACT_PATH,
    STAGE2_MODELS_PATH,
    CASCADE_THRESHOLDS_PATH,
    CASCADE_SUMMARY_PATH,
    CASCADE_METADATA_PATH,
    cascade_truth_path,
]:
    wandb.save(str(wandb_path))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved Stage 3 improved cascade results, trained models, thresholds, summary, metadata, reference profile, and cascade truth record.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        **cascade_stage3_improved_artifact_paths,
        "cascade_truth_hash": CASCADE_TRUTH_HASH,
        "cascade_truth_path": str(cascade_truth_path),
        "result_row_count": int(len(cascade_results)),
        "final_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "final_alert_count_test_rows": final_cascade_alert_count_test_rows,
        "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
        "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
        "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
    },
    logger=logger,
)

print("03C Stage 3 improved artifacts saved.")
print({
    "cascade_variant": CASCADE_VARIANT,
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "final_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "summary_path": str(CASCADE_SUMMARY_PATH),
    "metadata_path": str(CASCADE_METADATA_PATH),
})

{'dataset_name': 'pump',
 'cascade_variant': 'stage3_improved',
 'stage3_variant': 'tuned_confirmation_layer',
 'cascade_metrics': {'model': '3-Stage Cascade',
  'stage1_alert_count_all_rows': 103054,
  'stage2_alert_count_all_rows': 55684,
  'final_alert_count_all_rows': 37024,
  'stage1_alert_count_test_rows': 43236,
  'stage2_alert_count_test_rows': 16108,
  'final_alert_count_test_rows': 6594,
  'precision': 0.010615711252653927,
  'recall': 0.5932203389830508,
  'f1': 0.020858164481525627,
  'stage3_relaxed_precision': 0.005833880259607672,
  'stage3_relaxed_recall': 0.6779661016949152,
  'stage3_relaxed_f1': 0.01156821632564529,
  'stage3_relaxed_roc_auc': 0.8508693696637463,
  'stage3_relaxed_pr_auc': 0.10725219643480478,
  'stage3_relaxed_tn': 70138,
  'stage3_relaxed_fp': 13633,
  'stage3_relaxed_fn': 38,
  'stage3_relaxed_tp': 80,
  'stage3_relaxed_flag_column': 'cascade_stage3_relaxed_flag',
  'stage3_relaxed_score_column': 'stage3_weighted_score',
  'stage3_medium_precision

{'cascade_variant': 'stage3_improved',
 'stage2_selection_mode': 'parameter_search',
 'cascade_stage3_improved_results_path_csv': '/workspace/artifacts/gold/pump/cascade_stage3_improved/scores/pump__gold__cascade_stage3_improved_results.csv',
 'cascade_stage3_improved_results_path_pickle': '/workspace/artifacts/gold/pump/cascade_stage3_improved/scores/pump__gold__cascade_stage3_improved_results.pkl',
 'cascade_stage3_improved_stage1_model_artifact_path': '/workspace/artifacts/gold/pump/cascade_stage3_improved/models/pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib',
 'cascade_stage3_improved_stage1_models_path': '/workspace/models/pump/pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib',
 'cascade_stage3_improved_stage2_model_artifact_path': '/workspace/artifacts/gold/pump/cascade_stage3_improved/models/pump__gold__cascade_stage3_improved_stage2_isolation_forest.joblib',
 'cascade_stage3_improved_stage2_models_path': '/workspace/models/pump/pump__gold_

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,plot_order_index,stage1_threshold,stage1_threshold_percentile,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_strong_primary_flag,stage3_corroboration_flag,stage3_rule_evidence_count,stage3_weighted_evidence_score,stage3_weighted_score,stage3_confirmed_flag,cascade_stage3_improved_flag,cascade_final_flag,cascade_stage3_relaxed_flag,cascade_stage3_medium_flag,cascade_stage3_strict_flag,alert_priority,stage3_priority_class,stage3_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,14598431322315673869,run__001,sensor.csv,0,train,6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True,0.409368,0.090632,1,0,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,0,1,1,1.0,1.0,0,0,0,0,0,0,none,none,0
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,15954729095895098000,run__001,sensor.csv,1,train,6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True,0.409368,0.090632,1,0,1,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,0,1,1,1.0,1.0,0,0,0,0,0,0,none,none,0
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,10041703297090838359,run__001,sensor.csv,2,train,6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.280002,-0.565216,0.464286,1.307690,0.578311,-0.537436,-0.630443,-0.078128,0.859651,-0.666728,-1.160344,1.006862,0.146473,-0.108279,0.230324,-0.259305,-0.439206,-0.506023,0.044628,0.300410,0.019647,-0.099876,0.178490,0.225476,-0.356118,-0.460964,-0.049008,-1.791147,-0.158283,...,0.354679,0.470583,False,False,False,4,0,4,clean,True

,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-0.225564,1.983531,-2.540039,1.100022
1,sensor_01,0.0,0.021708,0.841543,-1.347824,1.456523
2,sensor_02,0.0,-0.011922,0.672431,-1.089288,1.017860
3,sensor_03,0.0,0.060602,0.691585,-0.980768,1.249997
4,sensor_04,0.0,-0.232387,1.951311,-1.927685,1.228905
5,sensor_05,0.0,0.015411,0.868819,-1.361052,1.536601
6,sensor_06,0.0,-0.038234,1.160401,-1.434801,1.804388
7,sensor_07,0.0,-0.164475,0.692269,-1.046894,1.156255
8,sensor_08,0.0,0.147561,0.763275,-0.929813,1.333325
9,sensor_09,0.0,-0.556567,3.136236,-2.666912,0.733364


2026-06-16 20:49:56,801 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_stage3_improved/thresholds/pump__gold__cascade_stage3_improved_thresholds.json
2026-06-16 20:49:56,862 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_stage3_improved/summaries/pump__gold__cascade_stage3_improved_summary.json
2026-06-16 20:49:56,917 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_stage3_improved/metadata/pump__gold__cascade_stage3_improved_metadata.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-06-16 20:49:57,668 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:49:57.668635+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'save_cascade_outputs', 'message': 'Saved Stage 3 improved cascade resul

03C Stage 3 improved artifacts saved.
{'cascade_variant': 'stage3_improved', 'cascade_truth_hash': '6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef0629ab072097d6805f', 'final_alert_count_test_rows': 6594, 'summary_path': '/workspace/artifacts/gold/pump/cascade_stage3_improved/summaries/pump__gold__cascade_stage3_improved_summary.json', 'metadata_path': '/workspace/artifacts/gold/pump/cascade_stage3_improved/metadata/pump__gold__cascade_stage3_improved_metadata.json'}


---

In [56]:
stage3_input_source = globals().get(
    "STAGE3_INPUT_SOURCE",
    "gold03c_recomputed_scores_or_existing_cascade_results",
)

stage3_contract_specs = [
    {
        "model_id": "stage3_improved",
        "model_label": "Stage 3 Improved",
        "operating_mode": "selected_improved",
        "flag_column": "cascade_final_flag",
        "metrics": cascade_metrics,
    },
    {
        "model_id": "stage3_relaxed",
        "model_label": "Stage 3 Relaxed",
        "operating_mode": "relaxed",
        "flag_column": stage3_operating_mode_metrics["relaxed"]["flag_column"],
        "metrics": stage3_operating_mode_metrics["relaxed"],
    },
    {
        "model_id": "stage3_medium",
        "model_label": "Stage 3 Medium",
        "operating_mode": "medium",
        "flag_column": stage3_operating_mode_metrics["medium"]["flag_column"],
        "metrics": stage3_operating_mode_metrics["medium"],
    },
    {
        "model_id": "stage3_strict",
        "model_label": "Stage 3 Strict",
        "operating_mode": "strict",
        "flag_column": stage3_operating_mode_metrics["strict"]["flag_column"],
        "metrics": stage3_operating_mode_metrics["strict"],
    },
]

stage3_contract_paths = []

for contract_spec in stage3_contract_specs:
    contract_path = gold_model_validation_contract_path(
        artifacts_root=paths.artifacts,
        dataset_id=DATASET_ID,
        model_id=contract_spec["model_id"],
    )

    contract = build_gold_model_output_validation_contract(
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        model_id=contract_spec["model_id"],
        model_label=contract_spec["model_label"],
        source_notebook="gold_03c_cascade_modeling",
        validation_type="stage3_rule_artifact",
        model_stage="cascade_stage3_improved_final",
        operating_mode=contract_spec["operating_mode"],
        metrics=contract_spec["metrics"],
        output_dataframe=cascade_results,
        output_flag_column=contract_spec["flag_column"],
        test_mask=test_mask,
        rule_config={
            "cascade_variant": CASCADE_VARIANT,
            "stage3_input_source": stage3_input_source,
            "stage3_selected_params": stage3_selected_params,
            "stage3_operating_mode_metrics": stage3_operating_mode_metrics,
            "stage3_variant_metric_summary": stage3_variant_metric_summary,
            "stage3_min_selection_recall": float(STAGE3_MIN_SELECTION_RECALL),
            "stage3_tuning_grid": STAGE3_TUNING_GRID,
            "stage3_search_candidate_count": int(len(stage3_search_results)),
            "relaxed_weighted_evidence_score_comparison": RELAXED_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON,
            "medium_weighted_evidence_score_comparison": MEDIUM_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON,
            "strict_weighted_evidence_score_comparison": STRICT_STAGE3_WEIGHTED_EVIDENCE_SCORE_COMPARISON,
        },
        rule_source="gold_03c_stage3_tuning_and_operating_mode_rule_cells",
        stage3_type="rule_based",
        stage3_saved_as_joblib=False,
        stage1_model_path=STAGE1_MODEL_ARTIFACT_PATH,
        stage2_model_path=STAGE2_MODEL_ARTIFACT_PATH,
        output_artifact_path=CASCADE_RESULTS_PATH_CSV,
        lineage_payload={
            "cascade_truth_hash": globals().get("CASCADE_TRUTH_HASH"),
            "parent_gold_truth_hash": globals().get("GOLD_PARENT_TRUTH_HASH"),
            "stage3_input_source": stage3_input_source,
        },
        notes="Gold 03c Stage 3 operating-mode contract. Stage 3 is rule-based and not saved as joblib.",
    )

    write_gold_model_output_validation_contract(
        contract=contract,
        output_path=contract_path,
    )

    stage3_contract_paths.append(contract_path)

    ledger.add(
        kind="artifact",
        step=f"gold03c_{contract['model_id']}_validation_contract_written",
        message="Wrote Gold 03c Stage 3 validation contract for Gold 06.",
        data={
            "path": str(contract_path),
            "model_id": contract["model_id"],
            "operating_mode": contract["operating_mode"],
            "alert_count_test_rows": contract["metric_payload"]["alert_count_test_rows"],
        },
        logger=logger,
    )

display(stage3_contract_paths)

2026-06-16 20:49:58,904 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:49:58.904586+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'artifact', 'step': 'gold03c_stage3_improved_validation_contract_written', 'message': 'Wrote Gold 03c Stage 3 validation contract for Gold 06.', 'why': None, 'consequence': None, 'data': {'path': '/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__stage3_improved_validation_contract.json', 'model_id': 'stage3_improved', 'operating_mode': 'selected_improved', 'alert_count_test_rows': 6594}}
2026-06-16 20:49:59,167 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:49:59.167228+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'artifact', 'step': 'gold03c_stage3_relaxed_validation_contract_written', 'message': 'Wrote Gold 03c Stage 3 validation contract for Gold 06.', 'why': None, 'consequence': None, 'dat

[PosixPath('/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__stage3_improved_validation_contract.json'),
 PosixPath('/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__stage3_relaxed_validation_contract.json'),
 PosixPath('/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__stage3_medium_validation_contract.json'),
 PosixPath('/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__stage3_strict_validation_contract.json')]

---

#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [57]:
stage1_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage1_flag",
    row_id_column="meta__row_id",
    score_column="stage1_score",
    decision_column="stage1_decision",
    pred_column="stage1_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage1_detected_rows",
    message="Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage1_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage1_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage1_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage1_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage1_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 1 detected row count: {len(stage1_detected_rows_dataframe):,}")
display(stage1_detected_rows_dataframe.head(20))

2026-06-16 20:50:00,419 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:50:00.418953+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage1_detected_rows', 'message': 'Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage1_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 103054, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 1 detected row count: 103,054


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage1_flag,stage1_score,stage1_decision,stage1_pred,stage2_raw_flag,stage2_flag,cascade_final_flag
0,260,13356824106279790836,260,260,2018-04-01 04:20:00+00:00,asset__001,run__001,NORMAL,0,1,0.493609,0.006391,1,0,0,0
1,262,14875396535299734039,262,262,2018-04-01 04:22:00+00:00,asset__001,run__001,NORMAL,0,1,0.493296,0.006704,1,0,0,0
2,265,9319389479859232848,265,265,2018-04-01 04:25:00+00:00,asset__001,run__001,NORMAL,0,1,0.500847,-0.000847,-1,0,0,0
3,266,7893936890144193503,266,266,2018-04-01 04:26:00+00:00,asset__001,run__001,NORMAL,0,1,0.496788,0.003212,1,0,0,0
4,267,7750815740698537469,267,267,2018-04-01 04:27:00+00:00,asset__001,run__001,NORMAL,0,1,0.496377,0.003623,1,0,0,0
5,268,17198309132295013087,268,268,2018-04-01 04:28:00+00:00,asset__001,run__001,NORMAL,0,1,0.498531,0.001469,1,0,0,0
6,269,13285358529408335145,269,269,2018-04-01 04:29:00+00:00,asset__001,run__001,NORMAL,0,1,0.494670,0.005330,1,0,0,0
7,270,8119318478283180666,270,270,2018-04-01 04:30:00+00:00,asset__001,run__001,NORMAL,0,1,0.500057,-0.000057,-1,0,0,0
8,271,2333674238325070709,271,271,2018-04-01 04:31:00+00:00,asset__001,run__001,NORMAL,0,1,0.500683,-0.000683,-1,0,0,0
9,272,18100572650003074832,272,272,2018-04-01 04:32:00+00:00,asset__001,run__001,NORMAL,0,1,0.496375,0.003625,1,0,0,0


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [58]:
stage2_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage2_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage2_detected_rows",
    message="Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage2_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage2_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage2_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage2_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage2_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 2 detected row count: {len(stage2_detected_rows_dataframe):,}")
display(stage2_detected_rows_dataframe.head(20))

2026-06-16 20:50:01,432 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:50:01.432452+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage2_detected_rows', 'message': 'Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage2_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 55684, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 2 detected row count: 55,684


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage2_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_model_score,stage2_model_flag,cascade_final_flag
0,5201,13712461203079535468,5201,5201,2018-04-04 14:41:00+00:00,asset__001,run__001,NORMAL,0,1,0.584908,-0.084908,-1.0,1,1,-0.584908,1.0,1
1,5202,12842405811221279513,5202,5202,2018-04-04 14:42:00+00:00,asset__001,run__001,NORMAL,0,1,0.587864,-0.087864,-1.0,1,1,-0.587864,1.0,1
2,5205,11598710423751654092,5205,5205,2018-04-04 14:45:00+00:00,asset__001,run__001,NORMAL,0,1,0.590463,-0.090463,-1.0,1,1,-0.590463,1.0,1
3,15018,8166674935408791495,15018,15018,2018-04-11 10:18:00+00:00,asset__001,run__001,NORMAL,0,1,0.589009,-0.089009,-1.0,1,1,-0.589009,1.0,0
4,16455,8939740980370716606,16455,16455,2018-04-12 10:15:00+00:00,asset__001,run__001,NORMAL,0,1,0.587093,-0.087093,-1.0,1,1,-0.587093,1.0,0
5,16457,2230535008236778778,16457,16457,2018-04-12 10:17:00+00:00,asset__001,run__001,NORMAL,0,1,0.585029,-0.085029,-1.0,1,1,-0.585029,1.0,0
6,16458,15015734555416799306,16458,16458,2018-04-12 10:18:00+00:00,asset__001,run__001,NORMAL,0,1,0.590556,-0.090556,-1.0,1,1,-0.590556,1.0,0
7,16461,5235549735633498512,16461,16461,2018-04-12 10:21:00+00:00,asset__001,run__001,NORMAL,0,1,0.584143,-0.084143,-1.0,1,1,-0.584143,1.0,0
8,16462,14349868644076285487,16462,16462,2018-04-12 10:22:00+00:00,asset__001,run__001,NORMAL,0,1,0.585232,-0.085232,-1.0,1,1,-0.585232,1.0,0
9,16477,18113483888750746465,16477,16477,2018-04-12 10:37:00+00:00,asset__001,run__001,NORMAL,0,1,0.610836,-0.110836,-1.0,1,1,-0.610836,1.0,0


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [59]:
stage3_evidence_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage3_profile_breach_flag",
    row_id_column="meta__row_id",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage3_evidence_rows",
    message="Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage3_profile_breach_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage3_evidence_rows_dataframe)),
    },
    logger=logger,
)

print(f"Stage 3 evidence row count: {len(stage3_evidence_rows_dataframe):,}")
display(stage3_evidence_rows_dataframe.head(20))

2026-06-16 20:50:02,841 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:50:02.841106+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage3_evidence_rows', 'message': 'Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage3_profile_breach_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 92627}}


Stage 3 evidence row count: 92,627


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage3_profile_breach_flag,stage1_flag,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count,cascade_final_flag
0,3,2072036352569063261,3,3,2018-04-01 00:03:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
1,4,4365040424004714369,4,4,2018-04-01 00:04:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
2,6,7206199315957152377,6,6,2018-04-01 00:06:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
3,8,4003627476009805685,8,8,2018-04-01 00:08:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
4,9,12843418223340544546,9,9,2018-04-01 00:09:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
5,10,12764708609319381380,10,10,2018-04-01 00:10:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,3,1,1,0,0,2,0
6,11,17692035958682023704,11,11,2018-04-01 00:11:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
7,12,9921446699215930665,12,12,2018-04-01 00:12:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
8,13,9345101619896030940,13,13,2018-04-01 00:13:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
9,363,9387970151524258740,363,363,2018-04-01 06:03:00+00:00,asset__001,run__001,NORMAL,0,1,1,0,0,2,1,1,0,0,2,0


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [60]:
final_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="cascade_final_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_final_detected_rows",
    message="Built the final detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "cascade_final_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(final_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in final_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in final_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in final_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Final cascade detected row count: {len(final_detected_rows_dataframe):,}")
display(final_detected_rows_dataframe.head(20))

2026-06-16 20:50:04,006 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:50:04.006265+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_final_detected_rows', 'message': 'Built the final detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'cascade_final_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 37024, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Final cascade detected row count: 37,024


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,cascade_final_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_flag,stage2_model_score,stage2_model_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count
0,5201,13712461203079535468,5201,5201,2018-04-04 14:41:00+00:00,asset__001,run__001,NORMAL,0,1,0.584908,-0.084908,-1.0,1,1,1,-0.584908,1.0,3,4,1,1,0,0,2
1,5202,12842405811221279513,5202,5202,2018-04-04 14:42:00+00:00,asset__001,run__001,NORMAL,0,1,0.587864,-0.087864,-1.0,1,1,1,-0.587864,1.0,3,4,1,1,0,0,2
2,5205,11598710423751654092,5205,5205,2018-04-04 14:45:00+00:00,asset__001,run__001,NORMAL,0,1,0.590463,-0.090463,-1.0,1,1,1,-0.590463,1.0,3,4,1,1,0,0,2
3,17872,17550074475687364978,17872,17872,2018-04-13 09:52:00+00:00,asset__001,run__001,RECOVERING,1,1,0.593469,-0.093469,-1.0,1,1,1,-0.593469,1.0,2,3,1,1,1,0,3
4,17879,7900973173592371794,17879,17879,2018-04-13 09:59:00+00:00,asset__001,run__001,RECOVERING,1,1,0.599508,-0.099508,-1.0,1,1,1,-0.599508,1.0,2,3,1,1,1,0,3
5,17880,11280900020643598843,17880,17880,2018-04-13 10:00:00+00:00,asset__001,run__001,RECOVERING,1,1,0.596768,-0.096768,-1.0,1,1,1,-0.596768,1.0,2,3,1,1,1,0,3
6,17881,13042761010288673424,17881,17881,2018-04-13 10:01:00+00:00,asset__001,run__001,RECOVERING,1,1,0.596768,-0.096768,-1.0,1,1,1,-0.596768,1.0,2,3,1,1,1,0,3
7,17884,1866828471338842541,17884,17884,2018-04-13 10:04:00+00:00,asset__001,run__001,RECOVERING,1,1,0.607504,-0.107504,-1.0,1,1,1,-0.607504,1.0,2,4,1,1,1,0,3
8,17885,1724819961315090964,17885,17885,2018-04-13 10:05:00+00:00,asset__001,run__001,RECOVERING,1,1,0.610846,-0.110846,-1.0,1,1,1,-0.610846,1.0,2,4,1,1,1,0,3
9,17886,9885301172107940696,17886,17886,2018-04-13 10:06:00+00:00,asset__001,run__001,RECOVERING,1,1,0.606999,-0.106999,-1.0,1,1,1,-0.606999,1.0,2,4,1,1,1,0,3


----

## Finalize the Ledger and Close the Tracking Run

This step writes the cascade ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling and artifact creation work is already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [ ]:
m,fgb
# =========================================================
# Finalize 03C ledger and close W&B run
# =========================================================

ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold 03C Stage 3 improved cascade modeling notebook complete.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "cascade_stage3_improved_metrics": cascade_metrics,
        **cascade_stage3_improved_artifact_paths,
        "cascade_truth_hash": CASCADE_TRUTH_HASH,
        "cascade_truth_path": str(cascade_truth_path),
        "comparison_ready": True,
    },
    logger=logger,
)

cascade_ledger_path = GOLD_CASCADE_ARTIFACT_DIRS["lineage"] / GOLD_CASCADE_STAGE3_IMPROVED_LEDGER_FILE_NAME
cascade_ledger_path.parent.mkdir(parents=True, exist_ok=True)

ledger.write_json(cascade_ledger_path)

wandb.save(str(cascade_ledger_path))
wandb_run.finish()

print("03C Stage 3 improved cascade modeling finalized.")
print({
    "ledger_path": str(cascade_ledger_path),
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "comparison_ready": True,
})

2026-06-16 20:50:04,774 | INFO | capstone.gold.cascade.stage3_improved | LEDGER | {'ts_utc': '2026-06-16T20:50:04.774858+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_cascade_modeling', 'message': 'Gold 03C Stage 3 improved cascade modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'cascade_variant': 'stage3_improved', 'stage2_selection_mode': 'parameter_search', 'cascade_stage3_improved_metrics': {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 103054, 'stage2_alert_count_all_rows': 55684, 'final_alert_count_all_rows': 37024, 'stage1_alert_count_test_rows': 43236, 'stage2_alert_count_test_rows': 16108, 'final_alert_count_test_rows': 6594, 'precision': 0.010615711252653927, 'recall': 0.5932203389830508, 'f1': 0.020858164481525627, 'stage3_relaxed_precision': 0.005833880259607672, 'stage3_relaxed_recall': 0.6779661016949152, 'stage3_relaxed_f1': 0.01156821632564529, 'stage3_relaxed_roc_auc': 0

03C Stage 3 improved cascade modeling finalized.
{'ledger_path': '/workspace/artifacts/gold/pump/cascade_stage3_improved/lineage/ledger__pump__gold_cascade_stage3_improved_modeling.json', 'cascade_truth_hash': '6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef0629ab072097d6805f', 'comparison_ready': True}


----

## Run Final Lineage and Consistency Checks

Before I treat the tuned cascade notebook as complete, I run a final sanity check on the saved cascade results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved cascade truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct cascade truth hash

I like ending with this because it confirms that the cascade output is not only saved, but also internally consistent and traceable.

In [62]:
required_cascade_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_cascade_meta_columns = [
    column_name
    for column_name in required_cascade_meta_columns
    if column_name not in cascade_results.columns
]
if missing_cascade_meta_columns:
    raise ValueError(
        f"cascade_results is missing required lineage columns: {missing_cascade_meta_columns}"
    )

cascade_results_truth_hash_check = extract_truth_hash(cascade_results)
if cascade_results_truth_hash_check is None:
    raise ValueError("cascade_results does not contain a readable meta__truth_hash value.")

if cascade_results_truth_hash_check != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_results truth hash does not match CASCADE_TRUTH_HASH:\n"
        f"dataframe={cascade_results_truth_hash_check}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

cascade_parent_values = cascade_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not cascade_parent_values:
    raise ValueError("cascade_results is missing populated meta__parent_truth_hash values.")

if len(cascade_parent_values) != 1:
    raise ValueError(f"cascade_results has multiple parent truth hashes: {cascade_parent_values}")

if cascade_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "cascade_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={cascade_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(cascade_truth_path).exists():
    raise FileNotFoundError(f"Cascade truth file was not created: {cascade_truth_path}")
loaded_cascade_truth: dict[str, Any] = require_mapping(
    load_json(cascade_truth_path),
    "loaded_cascade_truth",
)

loaded_cascade_truth_hash = loaded_cascade_truth.get("truth_hash")

if loaded_cascade_truth_hash != CASCADE_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file hash does not match CASCADE_TRUTH_HASH:\n"
        f"file={loaded_cascade_truth_hash}\n"
        f"expected={CASCADE_TRUTH_HASH}"
    )

loaded_cascade_parent_truth_hash = loaded_cascade_truth.get("parent_truth_hash")

if loaded_cascade_parent_truth_hash != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_cascade_parent_truth_hash}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_cascade_metadata: dict[str, Any] = require_mapping(
    load_json(CASCADE_METADATA_PATH),
    "saved_cascade_metadata",
)

saved_cascade_metadata_truth_hash = saved_cascade_metadata.get("cascade_truth_hash")

if saved_cascade_metadata_truth_hash != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_metadata cascade_truth_hash does not match CASCADE_TRUTH_HASH:\n"
        f"metadata={saved_cascade_metadata_truth_hash}\n"
        f"expected={CASCADE_TRUTH_HASH}"
    )

print("Gold Cascade lineage sanity check passed.")

2026-06-16 20:53:51,697 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_cascade/pump__gold_cascade__truth__6d55eabd515eda2c18cfe9218b59ded3e7f6c52ab277ef0629ab072097d6805f.json
2026-06-16 20:53:51,726 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/cascade_stage3_improved/metadata/pump__gold__cascade_stage3_improved_metadata.json


Gold Cascade lineage sanity check passed.


### Ask

What does this final sanity check really confirm?

### Answer

It confirms that the tuned cascade results can be trusted as a pipeline artifact, not just as notebook output.

A modeling notebook can run successfully and still leave behind broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check more than a completion check.

----

In [63]:
print("FULL cascade_metrics:", cascade_metrics)

FULL cascade_metrics: {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 103054, 'stage2_alert_count_all_rows': 55684, 'final_alert_count_all_rows': 37024, 'stage1_alert_count_test_rows': 43236, 'stage2_alert_count_test_rows': 16108, 'final_alert_count_test_rows': 6594, 'precision': 0.010615711252653927, 'recall': 0.5932203389830508, 'f1': 0.020858164481525627, 'stage3_relaxed_precision': 0.005833880259607672, 'stage3_relaxed_recall': 0.6779661016949152, 'stage3_relaxed_f1': 0.01156821632564529, 'stage3_relaxed_roc_auc': 0.8508693696637463, 'stage3_relaxed_pr_auc': 0.10725219643480478, 'stage3_relaxed_tn': 70138, 'stage3_relaxed_fp': 13633, 'stage3_relaxed_fn': 38, 'stage3_relaxed_tp': 80, 'stage3_relaxed_flag_column': 'cascade_stage3_relaxed_flag', 'stage3_relaxed_score_column': 'stage3_weighted_score', 'stage3_medium_precision': 0.010705462530881142, 'stage3_medium_recall': 0.6610169491525424, 'stage3_medium_f1': 0.02106969205834684, 'stage3_medium_roc_auc': 0.85086936966374

---

# Gold Cascade SQL Write Cell
Target:
- gold.anomaly_detection_scores

Purpose:
- Persist final cascade anomaly scores, flags, and rule/evidence fields.


In [64]:
WRITE_TO_POSTGRES = True

CASCADE_SQL_MODEL_STAGE = "cascade_stage3_improved_final"

if WRITE_TO_POSTGRES:
    gold_cascade_sql_summary_dataframe = write_gold_cascade_scores_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        notebook_globals=globals(),
        dataframe=cascade_results,
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
        model_stage=CASCADE_SQL_MODEL_STAGE,
    )

    display(gold_cascade_sql_summary_dataframe)

else:
    print("Postgres write skipped.")

Using supplied dataframe -> 220,320 rows x 374 columns
Deleted 220,320 existing rows from gold.anomaly_detection_scores for cascade_isolation_forest_rule_confirmation/cascade_stage3_improved_final.
Wrote 220,320 rows.
Upserted pipeline run metadata: run__001__gold_cascade_stage3_improved_final_modeling
Logged DQ event: gold.anomaly_detection_scores | cascade_stage3_improved_final_sql_write | pass


,model_name,model_stage,row_count,alert_count
0,cascade_isolation_forest_rule_confirmation,cascade_stage3_improved_final,220320,37024


In [65]:
verification_query = """
SELECT
    model_name,
    model_stage,
    COUNT(*) AS row_count,
    SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS alert_count
FROM gold.anomaly_detection_scores
WHERE dataset_id = :dataset_id
  AND run_id = :run_id
GROUP BY model_name, model_stage
ORDER BY model_name, model_stage
"""

score_stage_check_df = read_sql_dataframe(
    engine,
    verification_query,
    params={"dataset_id": DATASET_ID, "run_id": RUN_ID},
)

display(score_stage_check_df)

,model_name,model_stage,row_count,alert_count
0,baseline_isolation_forest,baseline,220320,87735
1,cascade_isolation_forest_rule_confirmation,cascade_default_final,220320,69362
2,cascade_isolation_forest_rule_confirmation,cascade_stage3_improved_final,220320,37024
3,cascade_isolation_forest_rule_confirmation,cascade_tuned_final,220320,54598


In [66]:
# Verification that notebooks are now writing safely without conflict

from utils.database.postgres import read_sql_dataframe, get_engine_from_env


gold03_stage_check_sql = """
SELECT
    model_name,
    model_stage,
    COUNT(*) AS row_count,
    SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS alert_count,
    MIN(meta_scored_at_utc) AS first_written_at,
    MAX(meta_scored_at_utc) AS last_written_at
FROM gold.anomaly_detection_scores
WHERE dataset_id = :dataset_id
  AND run_id = :run_id
  AND model_name = 'cascade_isolation_forest_rule_confirmation'
GROUP BY model_name, model_stage
ORDER BY model_stage;
"""

gold03_stage_check_df = read_sql_dataframe(
    engine,
    gold03_stage_check_sql,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(gold03_stage_check_df)

,model_name,model_stage,row_count,alert_count,first_written_at,last_written_at
0,cascade_isolation_forest_rule_confirmation,cascade_default_final,220320,69362,2026-06-16 18:20:36.709975+00:00,2026-06-16 18:20:36.709975+00:00
1,cascade_isolation_forest_rule_confirmation,cascade_stage3_improved_final,220320,37024,2026-06-16 20:56:23.740407+00:00,2026-06-16 20:56:23.740407+00:00
2,cascade_isolation_forest_rule_confirmation,cascade_tuned_final,220320,54598,2026-06-16 20:39:04.485562+00:00,2026-06-16 20:39:04.485562+00:00


In [67]:
'''

# Delete stale rows 

engine = get_engine_from_env()

from sqlalchemy import text

delete_stale_cascade_sql = """
DELETE FROM gold.anomaly_detection_scores
WHERE dataset_id = :dataset_id
  AND run_id = :run_id
  AND model_name = 'cascade_isolation_forest_rule_confirmation'
  AND model_stage = 'cascade_final';
"""

with engine.begin() as connection:
    result = connection.execute(
        text(delete_stale_cascade_sql),
        {
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

print(f"Deleted stale cascade_final rows: {result.rowcount:,}")
'''

'\n\n# Delete stale rows \n\nengine = get_engine_from_env()\n\nfrom sqlalchemy import text\n\ndelete_stale_cascade_sql = """\nDELETE FROM gold.anomaly_detection_scores\nWHERE dataset_id = :dataset_id\n  AND run_id = :run_id\n  AND model_name = \'cascade_isolation_forest_rule_confirmation\'\n  AND model_stage = \'cascade_final\';\n"""\n\nwith engine.begin() as connection:\n    result = connection.execute(\n        text(delete_stale_cascade_sql),\n        {\n            "dataset_id": DATASET_ID,\n            "run_id": RUN_ID,\n        },\n    )\n\nprint(f"Deleted stale cascade_final rows: {result.rowcount:,}")\n'

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Gold step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Gold 04 compares baseline and cascade outputs for final model selection evidence.
